In [ ]:
# ============================================================
# IMORT LIBRARies 
# ============================================================

import requests
import pandas as pd
import time
import logging
import os
from datetime import datetime, timedelta

# ─── BTC ────────────────────────────────────────────
TWELVE_DATA_API_KEY = ""
SYMBOL       = "BTC/USD"
SYMBOL_SAFE  = "BTCUSD"
CSV_RAW_FILE = f"{SYMBOL_SAFE}_5min_raw.csv"


TARGET_CANDLES = 500_000
MAX_REQUESTS   = 110       # 110 × 5000 = 550k شمعة حد أقصى

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[logging.StreamHandler()]
)


# ─── Fetcher ────────────────────────────────────────────────
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key):
        self.api_key = api_key

    def fetch_range(self, symbol=SYMBOL, interval="5min",
                    start_date=None, end_date=None, outputsize=5000):
        params = {
            'symbol':     symbol,
            'interval':   interval,
            'apikey':     self.api_key,
            'format':     'JSON',
            'order':      'ASC',
            'outputsize': outputsize,
        }
        if start_date:
            params['start_date'] = start_date
        if end_date:
            params['end_date'] = end_date

        try:
            r = requests.get(self.BASE_URL, params=params, timeout=30)
            if r.status_code != 200:
                logging.error(f"HTTP {r.status_code}: {r.text[:200]}")
                return pd.DataFrame()

            data = r.json()
            if 'values' not in data:
                msg = data.get('message', data.get('status', 'unknown error'))
                logging.error(f"API error: {msg}")
                return pd.DataFrame()

            df = pd.DataFrame(data['values'])

            if 'volume' not in df.columns:
                df['volume'] = 0

            for col in ['open', 'high', 'low', 'close', 'volume']:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

            df['datetime'] = pd.to_datetime(df['datetime'])
            df.sort_values('datetime', inplace=True)
            df.reset_index(drop=True, inplace=True)
            return df

        except requests.exceptions.Timeout:
            logging.error("Timeout — سيتم إعادة المحاولة")
            return pd.DataFrame()
        except Exception as e:
            logging.error(f"Fetch error: {e}")
            return pd.DataFrame()


# ─── جلب كامل التاريخ ───────────────────────────────────────
def fetch_full_history(target_candles=TARGET_CANDLES, interval="5min"):
    fetcher     = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    all_data    = []
    total       = 0
    end_date    = datetime.utcnow()
    prev_oldest = None
    retry_count = 0
    MAX_RETRIES = 3

    logging.info(f"⬇️  بدء جلب {target_candles:,} شمعة ({target_candles * 5 / 60 / 24 / 365:.1f} سنة)")

    for req in range(MAX_REQUESTS):
        start_date = end_date - timedelta(seconds=5000 * 300)
        batch = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=start_date.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=5000
        )

        if batch.empty:
            retry_count += 1
            if retry_count <= MAX_RETRIES:
                logging.warning(f"batch فاضي — محاولة {retry_count}/{MAX_RETRIES} بعد 5 ثواني")
                time.sleep(5)
                continue
            else:
                logging.warning("batch فاضي بعد كل المحاولات — توقف")
                break

        retry_count = 0
        all_data.append(batch)
        total   += len(batch)
        oldest   = batch['datetime'].min()
        newest   = batch['datetime'].max()

        # حفظ تدريجي كل 20 batch (لو انقطع الإنترنت)
        if req % 20 == 0 and req > 0:
            df_temp = pd.concat(all_data, ignore_index=True)
            df_temp.drop_duplicates(subset='datetime', keep='first', inplace=True)
            df_temp.sort_values('datetime', inplace=True)
            df_temp.to_csv(f"{CSV_RAW_FILE}.partial", index=False)
            logging.info(f"  💾 حفظ تدريجي: {len(df_temp):,} شمعة")

        logging.info(
            f"Batch {req+1:03d}: {len(batch):,} شمعة | "
            f"إجمالي: {total:,}/{target_candles:,} ({total/target_candles*100:.0f}%) | "
            f"من: {oldest.date()} → {newest.date()}"
        )

        if total >= target_candles:
            logging.info("✅ وصلنا للعدد المطلوب")
            break

        if prev_oldest is not None and prev_oldest == oldest:
            logging.info("⚠️  نفس أقدم تاريخ — وصلنا لحد البيانات المتاحة")
            break

        prev_oldest = oldest
        end_date    = oldest - timedelta(seconds=1)
        time.sleep(0.30)

    if not all_data:
        logging.error("❌ لم يتم جلب أي بيانات")
        return pd.DataFrame()

    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)

    if len(df) > target_candles:
        df = df.iloc[-target_candles:].reset_index(drop=True)

    # حذف الملف الـ partial لو موجود
    if os.path.exists(f"{CSV_RAW_FILE}.partial"):
        os.remove(f"{CSV_RAW_FILE}.partial")

    logging.info(f"\n✅ الداتا النهائية لـ BTC/USD:")
    logging.info(f"   {len(df):,} شمعة | {len(df) * 5 / 60 / 24 / 365:.2f} سنة")
    logging.info(f"   من: {df['datetime'].min().date()} → {df['datetime'].max().date()}")
    logging.info(f"   أول سعر: {df['close'].iloc[0]:,.2f}$ | آخر سعر: {df['close'].iloc[-1]:,.2f}$")
    return df


# ─── تحديث تدريجي ───────────────────────────────────────────
def update_data(csv_path=CSV_RAW_FILE, interval="5min"):
    if not os.path.exists(csv_path):
        logging.error(f"الملف {csv_path} غير موجود — استخدم fetch_full_history أولاً")
        return pd.DataFrame()

    df_old = pd.read_csv(csv_path)
    df_old['datetime'] = pd.to_datetime(df_old['datetime'])
    last_date = df_old['datetime'].max()

    logging.info(f"آخر شمعة محفوظة: {last_date}")
    logging.info("جلب الشموع الجديدة...")

    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    df_new  = fetcher.fetch_range(
        symbol=SYMBOL, interval=interval,
        start_date=(last_date + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
        end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
        outputsize=5000
    )

    if df_new.empty:
        logging.info("لا توجد شموع جديدة — الداتا محدّثة بالفعل ✅")
        return df_old

    df = pd.concat([df_old, df_new], ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='last', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv(csv_path, index=False)

    logging.info(
        f"✅ تحديث ناجح: +{len(df_new):,} شمعة جديدة | "
        f"الإجمالي: {len(df):,} | آخر: {df['datetime'].max()}"
    )
    return df


# ─── دالة موحّدة ────────────────────────────────────────────
def get_or_update_data(csv_path=CSV_RAW_FILE,
                       target_candles=TARGET_CANDLES,
                       force_refresh=False):
    colab_path = f"/content/{os.path.basename(csv_path)}"

    if not force_refresh:
        found_path = None
        if os.path.exists(csv_path):
            found_path = csv_path
        elif os.path.exists(colab_path):
            found_path = colab_path

        if found_path:
            df = pd.read_csv(found_path)
            df['datetime'] = pd.to_datetime(df['datetime'])
            logging.info(f"✅ CSV محمّل: {len(df):,} شمعة ({found_path})")

            # ★ v2.0: لو الداتا أقل من النصف المطلوب، اجلب من الأول
            if len(df) < target_candles * 0.5:
                logging.warning(
                    f"⚠️ الداتا الموجودة ({len(df):,}) أقل من 50% من المطلوب ({target_candles:,})\n"
                    f"   يُنصح بإعادة الجلب الكامل — استخدم force_refresh=True"
                )

            logging.info("🔄 تحديث بالشموع الجديدة...")
            df = update_data(found_path)
            return df

    logging.info(f"⬇️  جلب {target_candles:,} شمعة من API لـ BTC/USD...")
    df = fetch_full_history(target_candles=target_candles)
    if not df.empty:
        save_path = csv_path if os.access(os.path.dirname(csv_path) or '.', os.W_OK) else colab_path
        df.to_csv(save_path, index=False)
        logging.info(f"✅ محفوظ: {save_path}")
    return df


# ─── التحقق من جودة البيانات ────────────────────────────────
def validate_data(df: pd.DataFrame, target_candles=TARGET_CANDLES) -> bool:
    if df.empty:
        logging.error("❌ DataFrame فارغ")
        return False

    required_cols = ['datetime', 'open', 'high', 'low', 'close', 'volume']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        logging.error(f"❌ أعمدة مفقودة: {missing}")
        return False

    zero_close = (df['close'] <= 0).sum()
    if zero_close > 0:
        logging.warning(f"⚠️  {zero_close} شمعة بإغلاق صفر أو سالب")

    if not df['datetime'].is_monotonic_increasing:
        logging.warning("⚠️  البيانات غير مرتبة زمنياً")

    gap = df['datetime'].diff().dropna()
    expected_gap = pd.Timedelta(minutes=5)
    large_gaps = (gap > expected_gap * 5).sum()
    if large_gaps > 0:
        logging.info(f"ℹ️  {large_gaps} فجوة زمنية > 25 دقيقة")

    years_covered = len(df) * 5 / 60 / 24 / 365

    # ★ v2.0: تحقق من كفاية البيانات لكل فريم
    coverage_check = {
        '5min (4y needed)':  years_covered >= 4.0,
        '4h  (7y needed)':   years_covered >= 7.0,
        '1d  (10y needed)':  years_covered >= 10.0,
    }

    logging.info(f"✅ {len(df):,} شمعة | {years_covered:.2f} سنة | BTC: {df['close'].min():,.0f}$→{df['close'].max():,.0f}$")
    for check, ok in coverage_check.items():
        status = "✅" if ok else "⚠️ غير كافي"
        logging.info(f"   {status} {check}")

    pct_of_target = len(df) / target_candles * 100
    if pct_of_target < 80:
        logging.warning(f"⚠️ الداتا {pct_of_target:.0f}% من الهدف ({target_candles:,}) — الفريمات الكبيرة قد تعاني")

    return True


# ─── تشغيل مباشر ────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("  fetch_data_btc.py  v2.0  —  BTC/USD Data Fetcher")
    print(f"  Target: {TARGET_CANDLES:,} شمعة ≈ {TARGET_CANDLES * 5 / 60 / 24 / 365:.1f} سنوات")
    print("=" * 60)

    if os.path.exists(CSV_RAW_FILE):
        existing = pd.read_csv(CSV_RAW_FILE)
        print(f"\nالملف الموجود: {len(existing):,} شمعة")
        print(f"النسبة من الهدف: {len(existing)/TARGET_CANDLES*100:.0f}%")

        ans = input(
            f"\nاختر:\n"
            f"  [u] تحديث بالجديد فقط       (سريع)\n"
            f"  [r] إعادة جلب كامل 500k     (بطيء ~20-30 دقيقة)\n"
            f"  [v] التحقق من البيانات       (بدون تحميل)\n"
            f"  [n] إلغاء\n"
            f"اختيارك: "
        ).strip().lower()

        if ans == 'r':
            df = get_or_update_data(force_refresh=True)
        elif ans == 'u':
            df = update_data(CSV_RAW_FILE)
        elif ans == 'v':
            df = pd.read_csv(CSV_RAW_FILE)
            df['datetime'] = pd.to_datetime(df['datetime'])
            validate_data(df)
            print(f"\nالملف: {CSV_RAW_FILE}")
            print(df.tail(3).to_string())
            exit(0)
        else:
            print("تم الإلغاء.")
            exit(0)
    else:
        print(f"\nالملف '{CSV_RAW_FILE}' غير موجود — سيتم الجلب الكامل...")
        print(f"ده هياخد 20-30 دقيقة حسب سرعة الإنترنت")
        df = get_or_update_data(force_refresh=True)

    if not df.empty:
        validate_data(df)
        print(f"\n{'='*60}")
        print(f"✅ جاهز! {len(df):,} شمعة محفوظة في '{CSV_RAW_FILE}'")
        print(f"   الفترة: {df['datetime'].min().date()} → {df['datetime'].max().date()}")
        print(f"   التغطية: {len(df) * 5 / 60 / 24 / 365:.2f} سنة")
        print(f"   آخر سعر BTC: {df['close'].iloc[-1]:,.2f}$")
        print(f"{'='*60}")
        print(f"\nالخطوة التالية: شغّل train_patterns_btc_v6.py")
    else:
        print("❌ فشل جلب البيانات — تحقق من مفتاح API والاتصال بالإنترنت")

  fetch_data_btc.py  v2.0  —  BTC/USD Data Fetcher
  Target: 500,000 شمعة ≈ 4.8 سنوات

الملف 'BTCUSD_5min_raw.csv' غير موجود — سيتم الجلب الكامل...
ده هياخد 20-30 دقيقة حسب سرعة الإنترنت


/tmp/ipykernel_1380/135470280.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date    = datetime.utcnow()



✅ جاهز! 500,000 شمعة محفوظة في 'BTCUSD_5min_raw.csv'
   الفترة: 2021-07-06 → 2026-04-13
   التغطية: 4.76 سنة
   آخر سعر BTC: 71,812.01$

الخطوة التالية: شغّل train_patterns_btc_v6.py


In [9]:
pip install lightgbm

In [ ]:


# ============================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
import joblib, os, json, logging, warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
np.random.seed(42)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    logging.warning("LightGBM غير مثبّت — الـ ensemble هيشتغل بـ 3 موديلات فقط")


# ══════════════════════════════════════════════════════════════
# ثوابت
# ══════════════════════════════════════════════════════════════

SYMBOL       = "BTC/USD"
SYMBOL_SAFE  = "BTCUSD"
MODELS_DIR   = "models/btc_patterns"
CSV_RAW_FILE = "/content/BTCUSD_5min_raw.csv"

SPREAD_BPS   = 7
TAKER_FEE    = 0.0007

ATR_SL_MULT  = 1.8
ATR_TP_MULT  = 2.8

BTC_TP1_RATIO = 0.00001
BTC_TP2_RATIO = 0.000011
BTC_TP3_RATIO = 0.000012
BTC_TP4_RATIO = 0.000013
BTC_SL_RATIO  = 0.0000090

BTC_AVG_TP_RATIO = (BTC_TP1_RATIO + BTC_TP2_RATIO + BTC_TP3_RATIO + BTC_TP4_RATIO) / 4
BTC_RR_RATIO     = BTC_AVG_TP_RATIO / BTC_SL_RATIO
FIB_TOL = 0.07

# ★ v7.0: Circuit Breaker settings
MAX_CONSEC_LOSSES = 5
COOLING_BARS      = 10

# ★ v7.0: OOS محجوز
OOS_PCT = 0.10

MIN_TRADES_PER_FOLD = {
    "5min":  15,
    "15min": 12,
    "45min": 10,
    "1h":    8,
    "2h":    7,
    "4h":    6,
    "1d":    6,   # ★ v7.0: خُفِّض من 4 إلى 6 عشان نضمن consistency
    "1W":    3,
}


# ─────────────────────────────────────────────────────────────
# TIMEFRAME CONFIG — v7.0: تعديلات 1d و 1W وتحسينات عامة
# ─────────────────────────────────────────────────────────────
TIMEFRAME_CONFIG = {
    "5min": {
        "interval": "5min", "years": 4,
        "buy_threshold": 0.52, "sell_threshold": 0.48,
        "fold_buy_threshold": 0.50, "fold_sell_threshold": 0.50,
        "resample_rule": "5min", "minutes_per_candle": 5,
        "active_for_live": True, "min_candles": 600,
        "horizon": 12, "pattern_window": 30, "min_return": 0.0015,
        "htf_resample": "1h",
        "zigzag_pct": 0.007,
        "pattern_max_age": 20,           # ★ v7.0: 25→20 أنماط أحدث
        "confirmation_bars": 2, "higher_tf": "1h",
        "min_pattern_quality": 0.08,     # ★ v7.0: 0.10→0.08
        "confidence_threshold": 0.52,
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.2,
        "min_pattern_score": 0.15,       # ★ v7.0: جديد
        "time_filter_hours": None,       # لا فلتر زمني للـ 5min
    },
    "15min": {
        "interval": "15min", "years": 5,
        "buy_threshold": 0.52, "sell_threshold": 0.48,
        "fold_buy_threshold": 0.50, "fold_sell_threshold": 0.50,
        "resample_rule": "15min", "minutes_per_candle": 15,
        "active_for_live": True, "min_candles": 450,
        "horizon": 8, "pattern_window": 20, "min_return": 0.0025,
        "htf_resample": "4h",
        "zigzag_pct": 0.009,
        "pattern_max_age": 18,           # ★ v7.0: 22→18
        "confirmation_bars": 2, "higher_tf": "1h",
        "min_pattern_quality": 0.08,     # ★ v7.0: 0.10→0.08
        "confidence_threshold": 0.52,
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.3,
        "min_pattern_score": 0.20,       # ★ v7.0: جديد
        "time_filter_hours": None,
    },
    "45min": {
        "interval": "45min", "years": 5,
        "buy_threshold": 0.52, "sell_threshold": 0.48,
        "fold_buy_threshold": 0.50, "fold_sell_threshold": 0.50,
        "resample_rule": "45min", "minutes_per_candle": 45,
        "active_for_live": True, "min_candles": 250,
        "horizon": 6, "pattern_window": 15, "min_return": 0.0035,
        "htf_resample": "4h",
        "zigzag_pct": 0.010,
        "pattern_max_age": 15,           # ★ v7.0: 18→15
        "confirmation_bars": 2, "higher_tf": "4h",
        "min_pattern_quality": 0.08,     # ★ v7.0: 0.10→0.08
        "confidence_threshold": 0.51,
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.4,
        "min_pattern_score": 0.20,       # ★ v7.0: جديد
        "time_filter_hours": [7, 21],    # ★ v7.0: London+NY session
    },
    "1h": {
        "interval": "1h", "years": 6,
        "buy_threshold": 0.52, "sell_threshold": 0.48,
        "fold_buy_threshold": 0.50, "fold_sell_threshold": 0.50,
        "resample_rule": "1h", "minutes_per_candle": 60,
        "active_for_live": True, "min_candles": 350,
        "horizon": 5, "pattern_window": 15, "min_return": 0.0045,
        "htf_resample": "4h",
        "zigzag_pct": 0.011,
        "pattern_max_age": 15,           # ★ v7.0: 18→15
        "confirmation_bars": 3, "higher_tf": "4h",
        "min_pattern_quality": 0.10,     # ★ v7.0: 0.12→0.10
        "confidence_threshold": 0.52,
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.5,
        "min_pattern_score": 0.25,       # ★ v7.0: جديد
        "time_filter_hours": [7, 21],    # ★ v7.0: London+NY session
    },
    "2h": {
        "interval": "2h", "years": 6,
        "buy_threshold": 0.55,           # ★ v7.0: 0.52→0.55 لتحسين جودة الإشارة
        "sell_threshold": 0.45,          # ★ v7.0: 0.48→0.45
        "fold_buy_threshold": 0.52,      # ★ v7.0: 0.50→0.52
        "fold_sell_threshold": 0.48,     # ★ v7.0: 0.50→0.48
        "resample_rule": "2h", "minutes_per_candle": 120,
        "active_for_live": True, "min_candles": 180,
        "horizon": 4, "pattern_window": 12, "min_return": 0.005,
        "htf_resample": "1d",
        "zigzag_pct": 0.013,
        "pattern_max_age": 12,           # ★ v7.0: 15→12
        "confirmation_bars": 3, "higher_tf": "1d",
        "min_pattern_quality": 0.12,
        "confidence_threshold": 0.55,    # ★ v7.0: 0.52→0.55
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.5,
        "min_pattern_score": 0.30,       # ★ v7.0: جديد — أعلى threshold للـ 2h
        "time_filter_hours": [6, 20],
    },
    "4h": {
        "interval": "4h", "years": 7,
        "buy_threshold": 0.52, "sell_threshold": 0.48,
        "fold_buy_threshold": 0.50, "fold_sell_threshold": 0.50,
        "resample_rule": "4h", "minutes_per_candle": 240,
        "active_for_live": True, "min_candles": 120,
        "horizon": 3, "pattern_window": 12, "min_return": 0.007,
        "htf_resample": "1d",
        "zigzag_pct": 0.016,
        "pattern_max_age": 12,           # ★ v7.0: 14→12
        "confirmation_bars": 3, "higher_tf": "1d",
        "min_pattern_quality": 0.10,     # ★ v7.0: 0.12→0.10
        "confidence_threshold": 0.52,
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.6,
        "min_pattern_score": 0.25,       # ★ v7.0: جديد
        "time_filter_hours": None,
    },
    "1d": {
        "interval": "1d", "years": 12,   # ★ v7.0: 10→12 سنة
        "buy_threshold": 0.50,           # ★ v7.0: 0.51→0.50
        "sell_threshold": 0.50,          # ★ v7.0: 0.49→0.50
        "fold_buy_threshold": 0.50, "fold_sell_threshold": 0.50,
        "resample_rule": "1D", "minutes_per_candle": 1440,
        "active_for_live": True, "min_candles": 80,
        "horizon": 3, "pattern_window": 10, "min_return": 0.009,
        "htf_resample": "W-MON",
        "zigzag_pct": 0.012,             # ★ v7.0: 0.016→0.012 أنماط أكتر
        "pattern_max_age": 12,
        "confirmation_bars": 3, "higher_tf": "1W",
        "min_pattern_quality": 0.05,     # ★ v7.0: 0.10→0.05
        "confidence_threshold": 0.50,    # ★ v7.0: 0.51→0.50
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.0,          # ★ v7.0: 1.8→1.0 trades أكتر
        "min_pattern_score": 0.10,       # ★ v7.0: جديد
        "time_filter_hours": None,
        "use_htf_filter": False,         # ★ v7.0: جديد — الـ 1d هو HTF، لا تفلتر بـ 1W
    },
    "1W": {
        "interval": "1week", "years": 14,
        "buy_threshold": 0.50,           # ★ v7.0: 0.51→0.50
        "sell_threshold": 0.50,          # ★ v7.0: 0.49→0.50
        "fold_buy_threshold": 0.50, "fold_sell_threshold": 0.50,
        "resample_rule": "W-MON", "minutes_per_candle": 10080,
        "active_for_live": True, "min_candles": 40,
        "horizon": 3,                    # ★ v7.0: 2→3
        "pattern_window": 8, "min_return": 0.015,
        "htf_resample": None,
        "zigzag_pct": 0.012,             # ★ v7.0: 0.018→0.012
        "pattern_max_age": 10,
        "confirmation_bars": 2, "higher_tf": None,
        "min_pattern_quality": 0.03,     # ★ v7.0: 0.10→0.03
        "confidence_threshold": 0.50,    # ★ v7.0: 0.51→0.50
        "atr_sl_mult": ATR_SL_MULT, "atr_tp_mult": ATR_TP_MULT,
        "tp1_ratio": BTC_TP1_RATIO, "tp2_ratio": BTC_TP2_RATIO,
        "tp3_ratio": BTC_TP3_RATIO, "tp4_ratio": BTC_TP4_RATIO,
        "sl_ratio": BTC_SL_RATIO,
        "atr_target_mult": 1.2,          # ★ v7.0: 2.0→1.2
        "min_pattern_score": 0.05,       # ★ v7.0: جديد
        "time_filter_hours": None,
        "use_simple_split": True,        # ★ v7.0: بدل WF لأن البيانات قليلة
    },
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training_patterns_btc_v7.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)

try:
    import talib as _talib
    HAS_TALIB = True
except ImportError:
    HAS_TALIB = False


# ══════════════════════════════════════════════════════════════
# قراءة الداتا
# ══════════════════════════════════════════════════════════════
def get_or_update_data(force_refresh=False):
    candidates = [CSV_RAW_FILE, f"/content/{CSV_RAW_FILE}"]
    csv_path = None
    for p in candidates:
        if os.path.exists(p):
            csv_path = p
            break
    if csv_path is None:
        raise FileNotFoundError(
            f"❌ الملف '{CSV_RAW_FILE}' غير موجود!\n"
            "الحل: شغّل fetch_data_btc.py أولاً"
        )
    df = pd.read_csv(csv_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df[df['close'] > 0].copy()
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    logging.info(
        f"✅ CSV محمّل: {len(df):,} شمعة | "
        f"{df['datetime'].min().date()} → {df['datetime'].max().date()} ({csv_path})"
    )
    return df


# ══════════════════════════════════════════════════════════════
# ZIGZAG
# ══════════════════════════════════════════════════════════════
def zigzag_swings_non_repaint(high, low, pct_thresh=0.01, confirmation_bars=3):
    highs = high.values if hasattr(high, 'values') else np.asarray(high)
    lows  = low.values  if hasattr(low,  'values') else np.asarray(low)
    n = len(highs)
    if n < confirmation_bars + 3:
        return np.array([], dtype=int), np.array([], dtype=int)
    peak_idx, trough_idx = [], []
    direction = 1; last_high = 0; last_low = 0
    confirmed_idx_h = -1; confirmed_idx_l = -1
    for i in range(1, n):
        if direction == 1:
            if highs[i] > highs[last_high]:
                last_high = i
            if (lows[i] < highs[last_high] * (1.0 - pct_thresh)
                    and i - last_high >= confirmation_bars):
                if last_high > confirmed_idx_h:
                    peak_idx.append(last_high)
                    confirmed_idx_h = last_high
                direction = -1; last_low = i
        else:
            if lows[i] < lows[last_low]:
                last_low = i
            if (highs[i] > lows[last_low] * (1.0 + pct_thresh)
                    and i - last_low >= confirmation_bars):
                if last_low > confirmed_idx_l:
                    trough_idx.append(last_low)
                    confirmed_idx_l = last_low
                direction = 1; last_high = i
    return np.array(peak_idx, dtype=int), np.array(trough_idx, dtype=int)


def leg_ratio(a, b):
    return abs(b) / abs(a) if abs(a) > 1e-9 else 0.0


# ══════════════════════════════════════════════════════════════
# PATTERN FEATURE ENGINEER
# ══════════════════════════════════════════════════════════════
class PatternFeatureEngineer:
    def __init__(self):
        self.scaler          = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted   = False

    def add_base_features(self, df):
        df = df.copy()
        c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']
        n = len(df)
        p14  = min(14, max(3, n // 6))
        p20  = min(20, max(4, n // 5))
        p50  = min(50, max(5, n // 4))
        p200 = min(200, max(10, n // 2))

        df['ema_20']  = c.ewm(span=p20,  adjust=False).mean()
        df['ema_50']  = c.ewm(span=p50,  adjust=False).mean()
        df['ema_200'] = c.ewm(span=p200, adjust=False).mean()
        df['trend']   = np.sign(df['ema_20'] - df['ema_50'])
        df['ema_21']  = c.ewm(span=min(21, max(4, n // 5)), adjust=False).mean()
        df['ema_55']  = c.ewm(span=min(55, max(6, n // 4)), adjust=False).mean()
        df['ema_21_55_cross'] = np.sign(df['ema_21'] - df['ema_55'])

        # ★ v7.0: Bull/Bear market indicator
        df['bull_market'] = ((c > df['ema_200']) & (df['ema_50'] > df['ema_200'])).astype(float)
        df['bear_market'] = ((c < df['ema_200']) & (df['ema_50'] < df['ema_200'])).astype(float)

        tr = pd.concat([
            h - l,
            (h - c.shift(1)).abs(),
            (l - c.shift(1)).abs()
        ], axis=1).max(axis=1)
        df['atr']     = tr.rolling(p14).mean()
        df['atr_pct'] = df['atr'] / (c + 1e-9)

        delta = c.diff()
        ag = delta.clip(lower=0).rolling(p14).mean()
        al = (-delta.clip(upper=0)).rolling(p14).mean().clip(lower=1e-9)
        df['rsi'] = 100 - (100 / (1 + ag / al))

        df['rsi_slope']   = df['rsi'].diff(min(5, max(1, n // 20)))
        df['price_slope'] = c.pct_change(min(5, max(1, n // 20)))
        df['rsi_divergence'] = np.where(
            (df['rsi_slope'] > 0) & (df['price_slope'] < 0), 1.0,
            np.where((df['rsi_slope'] < 0) & (df['price_slope'] > 0), -1.0, 0.0)
        )

        ema12 = c.ewm(span=min(12, max(3, n // 8)), adjust=False).mean()
        ema26 = c.ewm(span=min(26, max(5, n // 5)), adjust=False).mean()
        df['macd']      = ema12 - ema26
        df['macd_sig']  = df['macd'].ewm(span=min(9, max(3, n // 10)), adjust=False).mean()
        df['macd_hist'] = df['macd'] - df['macd_sig']

        sma20 = c.rolling(p20).mean()
        std20 = c.rolling(p20).std()
        df['bb_upper'] = sma20 + 2 * std20
        df['bb_lower'] = sma20 - 2 * std20
        df['bb_pctb']  = (c - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-9)
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (sma20 + 1e-9)

        kc_mid  = c.ewm(span=p20, adjust=False).mean()
        kc_mult = 1.5
        df['kc_upper']   = kc_mid + kc_mult * df['atr']
        df['kc_lower']   = kc_mid - kc_mult * df['atr']
        df['kc_squeeze'] = (df['bb_upper'] < df['kc_upper']).astype(float)

        typ_p  = (h + l + c) / 3
        raw_mf = typ_p * v
        mf_dir = np.where(typ_p > typ_p.shift(1), 1, -1)
        pos_mf = (raw_mf * (mf_dir > 0)).rolling(p14).sum()
        neg_mf = (raw_mf * (mf_dir < 0)).rolling(p14).sum()
        mfr    = pos_mf / (neg_mf + 1e-9)
        df['mfi'] = 100 - (100 / (1 + mfr))

        rsi_min = df['rsi'].rolling(p14).min()
        rsi_max = df['rsi'].rolling(p14).max()
        df['stoch_rsi']   = (df['rsi'] - rsi_min) / (rsi_max - rsi_min + 1e-9)
        df['stoch_rsi_k'] = df['stoch_rsi'].rolling(min(3, max(1, n // 20))).mean()
        df['stoch_rsi_d'] = df['stoch_rsi_k'].rolling(min(3, max(1, n // 20))).mean()

        vwap_window = min(48, max(5, n // 6))
        vol_ma_s    = v.rolling(p20).mean().shift(1).fillna(v.rolling(p20).mean())
        df['vol_ratio'] = v / (vol_ma_s + 1e-9)
        df['vol_surge'] = (df['vol_ratio'] > 2.5).astype(float)
        df['vol_ema']   = v.ewm(span=p20, adjust=False).mean()
        df['vol_trend'] = np.sign(df['vol_ema'].diff(min(5, max(1, n // 20))))

        tp_vwap    = (h + l + c) / 3
        df['vwap'] = ((tp_vwap * v).rolling(vwap_window, min_periods=1).sum()
                     / (v.rolling(vwap_window, min_periods=1).sum() + 1e-9))
        df['dist_vwap'] = (c - df['vwap']) / (df['atr'] + 1e-9)

        obv = (np.sign(c.diff()) * v).fillna(0).cumsum()
        s5  = min(5, max(1, n // 15))
        df['obv_slope']      = obv.diff(s5) / (obv.shift(s5).abs() + 1e-9)
        df['obv_slope_norm'] = df['obv_slope'] / (df['obv_slope'].rolling(p20).std() + 1e-9)

        df['returns']    = c.pct_change()
        df['volatility'] = df['returns'].rolling(p20).std()
        df['body_pct']   = (c - o).abs() / (h - l + 1e-9)
        df['upper_wick'] = (h - c.clip(upper=o)) / (h - l + 1e-9)
        df['lower_wick'] = (c.clip(lower=o) - l) / (h - l + 1e-9)
        df['body_size_ma'] = df['body_pct'].rolling(10).mean()
        df['wick_ratio']   = (df['upper_wick'] + df['lower_wick']) / (df['body_pct'] + 1e-9)

        df['hour']     = df['datetime'].dt.hour
        df['dow']      = df['datetime'].dt.dayofweek
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

        df['is_us']     = ((df['hour'] >= 13) & (df['hour'] <= 21)).astype(float)
        df['is_asia']   = ((df['hour'] >= 0)  & (df['hour'] <= 8)).astype(float)
        df['is_europe'] = ((df['hour'] >= 7)  & (df['hour'] <= 15)).astype(float)
        df['is_overlap_us_eu'] = ((df['hour'] >= 13) & (df['hour'] <= 15)).astype(float)
        df['is_monday']  = (df['dow'] == 0).astype(float)
        df['is_friday']  = (df['dow'] == 4).astype(float)
        df['is_weekend'] = (df['dow'] >= 5).astype(float)

        sr_window = min(50, max(5, n // 4))
        res_s = h.shift(1).rolling(sr_window, min_periods=1).max()
        sup_s = l.shift(1).rolling(sr_window, min_periods=1).min()
        df['dist_to_resistance'] = (res_s - c) / (df['atr'] + 1e-9)
        df['dist_to_support']    = (c - sup_s) / (df['atr'] + 1e-9)
        df['near_breakout']      = (
            (df['dist_to_resistance'] < 1.5) | (df['dist_to_support'] < 1.5)
        ).astype(float)
        df['sr_position'] = (c - sup_s) / (res_s - sup_s + 1e-9)

        vol_rank = df['atr_pct'].rolling(min(100, max(10, n // 3)), min_periods=5).rank(pct=True)
        df['vol_regime_btc'] = np.select(
            [vol_rank < 0.33, vol_rank > 0.66], [0, 2], default=1
        )

        df['funding_proxy'] = df['vol_ratio'] * df['atr_pct'] * np.sign(df['returns'])
        df['momentum_7']    = c.pct_change(min(7,  max(1, n // 15)))
        df['momentum_14']   = c.pct_change(min(14, max(2, n // 12)))
        df['momentum_30']   = c.pct_change(min(30, max(3, n // 10)))

        df['higher_high'] = ((h > h.shift(1)) & (l > l.shift(1))).astype(float)
        df['lower_low']   = ((h < h.shift(1)) & (l < l.shift(1))).astype(float)
        df['inside_bar']  = ((h <= h.shift(1)) & (l >= l.shift(1))).astype(float)
        df['outside_bar'] = ((h > h.shift(1)) & (l < l.shift(1))).astype(float)
        df['hh_count']    = df['higher_high'].rolling(5).sum()
        df['ll_count']    = df['lower_low'].rolling(5).sum()

        # ★ v7.0: market structure score
        df['market_structure_score'] = (
            (df['hh_count'] - df['ll_count']) / 5.0
        ).clip(-1, 1)

        return df

    def add_htf_trend(self, df, htf_resample=None):
        df = df.copy()
        htf_cols = ['htf_trend', 'htf_ema_slope', 'htf_rsi', 'htf_above_ema200',
                    'htf_volatility', 'htf_dist_ema20']
        for col in htf_cols:
            df[col] = 0.0
        if htf_resample is None:
            return df
        try:
            df_s = df.sort_values('datetime').reset_index(drop=True)
            df_s['datetime'] = pd.to_datetime(df_s['datetime'])
            htf = (df_s.set_index('datetime')
                   .resample(htf_resample)
                   .agg({'open': 'first', 'high': 'max',
                         'low': 'min', 'close': 'last', 'volume': 'sum'})
                   .dropna())
            if len(htf) < 5:
                return df
            htf = htf.reset_index()
            if 'datetime' not in htf.columns:
                htf = htf.rename(columns={htf.columns[0]: 'datetime'})
            htf['datetime'] = pd.to_datetime(htf['datetime'])
            c_h = htf['close']
            nh  = len(htf)
            sp20  = min(20,  max(4, nh // 5))
            sp50  = min(50,  max(5, nh // 4))
            sp200 = min(200, max(10, nh // 2))
            htf['e20']  = c_h.ewm(span=sp20,  adjust=False).mean()
            htf['e50']  = c_h.ewm(span=sp50,  adjust=False).mean()
            htf['e200'] = c_h.ewm(span=sp200, adjust=False).mean()
            htf['htf_trend']        = np.sign(htf['e50'] - htf['e200'])
            htf['htf_ema_slope']    = htf['e50'].diff(min(3, max(1, nh // 15))) / (htf['e50'].shift(3) + 1e-9)
            htf['htf_above_ema200'] = (c_h > htf['e200']).astype(float)
            htf['htf_dist_ema20']   = (c_h - htf['e20']) / (c_h.rolling(sp20).std() + 1e-9)
            htf['htf_volatility']   = c_h.pct_change().rolling(sp20).std()
            p14h = min(14, max(3, nh // 6))
            d    = c_h.diff()
            g    = d.clip(lower=0).rolling(p14h).mean()
            ls   = (-d.clip(upper=0)).rolling(p14h).mean().clip(lower=1e-9)
            htf['htf_rsi'] = 100 - (100 / (1 + g / ls))
            slim = htf[['datetime'] + htf_cols].copy().sort_values('datetime').reset_index(drop=True)
            merged = pd.merge_asof(df_s, slim, on='datetime', direction='backward')
            for col in htf_cols:
                if col in merged.columns:
                    df_s[col] = merged[col].fillna(0).values
            for col in htf_cols:
                df[col] = df_s[col].fillna(0).values
        except Exception as e:
            logging.warning(f"HTF error: {e}")
        return df

    def add_regime_features(self, df):
        df = df.copy()
        n  = len(df)
        if HAS_TALIB:
            try:
                df['adx'] = _talib.ADX(
                    df['high'].values, df['low'].values,
                    df['close'].values, timeperiod=14
                )
            except Exception:
                df['adx'] = (df['atr'] / (df['close'] + 1e-9) * 500).clip(0, 100)
        else:
            df['adx'] = (df['atr'] / (df['close'] + 1e-9) * 500).clip(0, 100)

        vol_win  = min(100, max(10, n // 3))
        atr_rank = df['atr'].rolling(vol_win, min_periods=5).rank(pct=True)
        df['volatility_regime'] = np.select(
            [atr_rank < 0.33, atr_rank > 0.66], [0, 2], default=1
        )
        df['trend_strength']  = df['adx'] / 50.0
        df['market_regime']   = (df['adx'] > 20).astype(float)
        df['year_in_cycle']   = (df['datetime'].dt.year - 2012) % 4
        df['cycle_phase']     = df['year_in_cycle'] / 4.0

        df['strong_trend']    = (df['adx'] > 30).astype(float)
        df['ranging_market']  = (df['adx'] < 15).astype(float)
        return df

    def add_order_flow(self, df):
        df = df.copy()
        n       = len(df)
        cmf_win = min(21, max(5, n // 8))
        mfv = (
            ((df['close'] - df['low']) - (df['high'] - df['close']))
            / (df['high'] - df['low'] + 1e-9)
            * df['volume']
        )
        df['cmf'] = (mfv.rolling(cmf_win).sum()
                    / (df['volume'].rolling(cmf_win).sum() + 1e-9))

        buy_vol  = df['volume'] * ((df['close'] - df['low'])  / (df['high'] - df['low'] + 1e-9))
        sell_vol = df['volume'] * ((df['high'] - df['close']) / (df['high'] - df['low'] + 1e-9))
        df['cvd_proxy'] = (buy_vol - sell_vol).rolling(cmf_win).sum()
        df['cvd_slope'] = df['cvd_proxy'].diff(min(5, max(1, n // 15)))
        df['cvd_norm']  = df['cvd_proxy'] / (df['volume'].rolling(cmf_win).sum() + 1e-9)
        return df

    # ══════════════════════════════════════════════════════════
    # CLASSIC PATTERNS
    # ══════════════════════════════════════════════════════════
    def detect_classic_patterns(self, df, window=20, zigzag_pct=0.01,
                                 max_age=20, confirmation_bars=3):
        df = df.copy()
        n  = len(df)
        hi = df['high'].values
        lo = df['low'].values
        cl = df['close'].values
        vl = df['volume'].values

        cols = [
            'hs_pattern', 'hs_direction', 'hs_strength', 'hs_valid',
            'double_pattern', 'double_direction', 'double_strength', 'double_valid',
            'triple_pattern', 'triple_direction', 'triple_strength', 'triple_valid',
            'triangle_pattern', 'triangle_direction', 'triangle_strength',
            'flag_pattern', 'flag_direction', 'flag_strength',
            'wedge_pattern', 'wedge_direction', 'wedge_strength',
            'channel_pattern', 'channel_direction', 'channel_strength',
            'cup_handle_pattern', 'cup_handle_strength', 'cup_handle_valid',
            'rectangle_pattern', 'rectangle_direction', 'rectangle_strength',
            'pattern_age', 'pattern_volume_confirm'
        ]
        for col in cols:
            df[col] = 0.0

        vol_s  = pd.Series(vl)
        vol_ma = (vol_s.rolling(20, min_periods=3).mean()
                  .shift(1)
                  .fillna(vol_s.rolling(20, min_periods=3).mean())
                  .values)
        atr_v  = df['atr'].values

        hi_idx, lo_idx = zigzag_swings_non_repaint(
            df['high'], df['low'],
            pct_thresh=zigzag_pct, confirmation_bars=confirmation_bars
        )
        swings = sorted(
            [{'idx': i, 'price': hi[i], 'type': 'H'} for i in hi_idx]
            + [{'idx': i, 'price': lo[i], 'type': 'L'} for i in lo_idx],
            key=lambda x: x['idx']
        )

        for si in range(2, len(swings)):
            pts    = swings[:si + 1]
            i      = pts[-1]['idx']
            if i >= n or i + 1 >= n:
                continue
            a      = max(atr_v[i], 1e-9)
            types  = [p['type'] for p in pts]
            prices = [p['price'] for p in pts]
            vc     = float(vl[i] / (vol_ma[i] + 1e-9) > 2.0)

            if len(pts) >= 5:
                t5 = types[-5:]; p5 = prices[-5:]
                if t5 == ['H', 'L', 'H', 'L', 'H']:
                    sh_l, neck_l, head, neck_r, sh_r = p5
                    neck = (neck_l + neck_r) / 2
                    if head > sh_l * 1.008 and head > sh_r * 1.008:
                        s = min(1.0, (head - max(sh_l, sh_r)) / (a * 3)) * (1.4 if vc else 0.7)
                        df.at[i, 'hs_pattern']   = 1.0
                        df.at[i, 'hs_direction'] = -1.0
                        df.at[i, 'hs_strength']  = min(1.0, s)
                        df.at[i, 'hs_valid']     = float(cl[i] < neck)
                elif t5 == ['L', 'H', 'L', 'H', 'L']:
                    sh_l, neck_l, head, neck_r, sh_r = p5
                    neck = (neck_l + neck_r) / 2
                    if head < sh_l * 0.992 and head < sh_r * 0.992:
                        s = min(1.0, (min(sh_l, sh_r) - head) / (a * 3)) * (1.4 if vc else 0.7)
                        df.at[i, 'hs_pattern']   = 1.0
                        df.at[i, 'hs_direction'] = 1.0
                        df.at[i, 'hs_strength']  = min(1.0, s)
                        df.at[i, 'hs_valid']     = float(cl[i] > neck)

            if len(pts) >= 3:
                t3 = types[-3:]; p3 = prices[-3:]
                if t3 == ['H', 'L', 'H']:
                    h1, valley, h2 = p3
                    if abs(h1 - h2) / a < 2.0:
                        s = (1.0 - abs(h1 - h2) / (a * 2.0 + 1e-9)) * (1.3 if vc else 0.7)
                        df.at[i, 'double_pattern']   = 1.0
                        df.at[i, 'double_direction'] = -1.0
                        df.at[i, 'double_strength']  = min(1.0, s)
                        df.at[i, 'double_valid']     = float(cl[i] < valley)
                elif t3 == ['L', 'H', 'L']:
                    l1, peak, l2 = p3
                    if abs(l1 - l2) / a < 2.0:
                        s = (1.0 - abs(l1 - l2) / (a * 2.0 + 1e-9)) * (1.3 if vc else 0.7)
                        df.at[i, 'double_pattern']   = 1.0
                        df.at[i, 'double_direction'] = 1.0
                        df.at[i, 'double_strength']  = min(1.0, s)
                        df.at[i, 'double_valid']     = float(cl[i] > peak)

            if len(pts) >= 5:
                t5 = types[-5:]; p5 = prices[-5:]
                if t5 == ['H', 'L', 'H', 'L', 'H']:
                    h1, v1, h2, v2, h3 = p5
                    if abs(h1 - h2) / a < 2.0 and abs(h2 - h3) / a < 2.0:
                        neck = min(v1, v2)
                        s = max(0.0, 1.0 - (abs(h1 - h2) + abs(h2 - h3)) / (a * 4)) * (1.3 if vc else 0.7)
                        df.at[i, 'triple_pattern']   = 1.0
                        df.at[i, 'triple_direction'] = -1.0
                        df.at[i, 'triple_strength']  = min(1.0, s)
                        df.at[i, 'triple_valid']     = float(cl[i] < neck)
                elif t5 == ['L', 'H', 'L', 'H', 'L']:
                    l1, p1, l2, p2, l3 = p5
                    if abs(l1 - l2) / a < 2.0 and abs(l2 - l3) / a < 2.0:
                        neck = max(p1, p2)
                        s = max(0.0, 1.0 - (abs(l1 - l2) + abs(l2 - l3)) / (a * 4)) * (1.3 if vc else 0.7)
                        df.at[i, 'triple_pattern']   = 1.0
                        df.at[i, 'triple_direction'] = 1.0
                        df.at[i, 'triple_strength']  = min(1.0, s)
                        df.at[i, 'triple_valid']     = float(cl[i] > neck)

            lh2 = [p for p in pts if p['type'] == 'H'][-2:]
            ll2 = [p for p in pts if p['type'] == 'L'][-2:]
            if len(lh2) == 2 and len(ll2) == 2:
                h_dx = max(1, lh2[-1]['idx'] - lh2[-2]['idx'])
                l_dx = max(1, ll2[-1]['idx'] - ll2[-2]['idx'])
                h_sl = (lh2[-1]['price'] - lh2[-2]['price']) / h_dx
                l_sl = (ll2[-1]['price'] - ll2[-2]['price']) / l_dx
                if h_sl < -1e-6 and l_sl > 1e-6:
                    df.at[i, 'triangle_pattern']   = 1.0
                    df.at[i, 'triangle_direction'] = 0.0
                    df.at[i, 'triangle_strength']  = 0.7 * (1.2 if vc else 0.8)
                elif abs(h_sl) < 1e-6 and l_sl > 1e-6:
                    df.at[i, 'triangle_pattern']   = 1.0
                    df.at[i, 'triangle_direction'] = 1.0
                    df.at[i, 'triangle_strength']  = 0.80 * (1.3 if vc else 0.7)
                elif h_sl < -1e-6 and abs(l_sl) < 1e-6:
                    df.at[i, 'triangle_pattern']   = 1.0
                    df.at[i, 'triangle_direction'] = -1.0
                    df.at[i, 'triangle_strength']  = 0.80 * (1.3 if vc else 0.7)
                elif h_sl > 1e-6 and l_sl > 1e-6 and l_sl > h_sl:
                    df.at[i, 'wedge_pattern']   = 1.0
                    df.at[i, 'wedge_direction'] = -1.0
                    df.at[i, 'wedge_strength']  = min(1.0, abs(l_sl - h_sl) / a * 100 * (1.3 if vc else 0.7))
                elif h_sl < -1e-6 and l_sl < -1e-6 and l_sl > h_sl:
                    df.at[i, 'wedge_pattern']   = 1.0
                    df.at[i, 'wedge_direction'] = 1.0
                    df.at[i, 'wedge_strength']  = min(1.0, abs(h_sl - l_sl) / a * 100 * (1.3 if vc else 0.7))
                sd = abs(h_sl - l_sl) / a
                if (sd < 0.3
                        and df.at[i, 'triangle_pattern'] == 0
                        and df.at[i, 'wedge_pattern'] == 0):
                    df.at[i, 'channel_pattern']   = 1.0
                    df.at[i, 'channel_direction'] = (1.0 if h_sl > 0 else (-1.0 if h_sl < 0 else 0.0))
                    df.at[i, 'channel_strength']  = max(0.0, 1.0 - sd) * (1.1 if vc else 0.9)

            if i >= window:
                pm = abs(cl[i] - cl[i - window]) / a
                rr = (max(hi[max(0, i - 5):i + 1]) - min(lo[max(0, i - 5):i + 1])) / a
                if pm > 6 and rr < 2.5:
                    df.at[i, 'flag_pattern']   = 1.0
                    df.at[i, 'flag_direction'] = 1.0 if cl[i] > cl[i - window] else -1.0
                    df.at[i, 'flag_strength']  = min(1.0, pm / 12) * (1.3 if vc else 0.7)

            if len(pts) >= 3:
                t3 = types[-3:]; p3 = prices[-3:]
                if t3 == ['H', 'L', 'H']:
                    lh_c, cup_l, rh_c = p3
                    depth = (min(lh_c, rh_c) - cup_l) / a
                    rdiff = abs(lh_c - rh_c) / a
                    if depth > 4 and rdiff < 2.5 and cl[i] > cup_l * 1.01:
                        rim = max(lh_c, rh_c)
                        df.at[i, 'cup_handle_pattern']  = 1.0
                        df.at[i, 'cup_handle_strength'] = min(1.0, depth / 12) * (1.4 if vc else 0.7)
                        df.at[i, 'cup_handle_valid']    = float(cl[i] > rim)

            l3h = [p['price'] for p in pts if p['type'] == 'H'][-3:]
            l3l = [p['price'] for p in pts if p['type'] == 'L'][-3:]
            if len(l3h) == 3 and len(l3l) == 3:
                hs2 = np.std(l3h); ls2 = np.std(l3l)
                ha  = np.mean(l3h); la  = np.mean(l3l)
                if hs2 / a < 1.5 and ls2 / a < 1.5 and la < cl[i] < ha:
                    df.at[i, 'rectangle_pattern']   = 1.0
                    df.at[i, 'rectangle_direction'] = 0.0
                    df.at[i, 'rectangle_strength']  = max(0.0, 1.0 - (hs2 + ls2) / (a * 3 + 1e-9))

            df.at[i, 'pattern_volume_confirm'] = vc

        has_any = (
            df['hs_pattern'] + df['double_pattern'] + df['triple_pattern']
            + df['triangle_pattern'] + df['wedge_pattern'] + df['flag_pattern']
            + df['channel_pattern'] + df['cup_handle_pattern'] + df['rectangle_pattern']
        ) > 0
        df['pattern_intensity'] = (
            df['hs_pattern'] * 1.5 + df['double_pattern'] + df['triple_pattern'] * 1.2
            + df['triangle_pattern'] * 0.8 + df['wedge_pattern'] * 0.9 + df['flag_pattern'] * 1.8
            + df['channel_pattern'] * 0.5 + df['cup_handle_pattern'] * 1.3 + df['rectangle_pattern'] * 0.4
        )
        count = 999; bars = []
        for v_ in has_any:
            count = 0 if v_ else min(count + 1, 999)
            bars.append(count)
        df['pattern_age'] = bars
        return df

    # ══════════════════════════════════════════════════════════
    # HARMONIC PATTERNS
    # ══════════════════════════════════════════════════════════
    def detect_harmonic_patterns(self, df, zigzag_pct=0.01, confirmation_bars=3):
        df = df.copy()
        n  = len(df)
        for col in ['harmonic_pattern', 'harmonic_type', 'harmonic_direction',
                    'harmonic_strength', 'harmonic_prz', 'harmonic_prz_high',
                    'harmonic_prz_low', 'harmonic_valid']:
            df[col] = 0.0

        PATS = {
            'gartley':   {'AB_XA': (0.618, 0.618), 'BC_AB': (0.382, 0.886), 'CD_BC': (1.272, 1.618), 'AD_XA': (0.786, 0.786), 'sc': 4, 'prz_fib': 0.786},
            'butterfly': {'AB_XA': (0.786, 0.786), 'BC_AB': (0.382, 0.886), 'CD_BC': (1.618, 2.618), 'AD_XA': (1.272, 1.618), 'sc': 3, 'prz_fib': 1.272},
            'bat':       {'AB_XA': (0.382, 0.500), 'BC_AB': (0.382, 0.886), 'CD_BC': (1.618, 2.618), 'AD_XA': (0.886, 0.886), 'sc': 3, 'prz_fib': 0.886},
            'crab':      {'AB_XA': (0.382, 0.618), 'BC_AB': (0.382, 0.886), 'CD_BC': (2.240, 3.618), 'AD_XA': (1.618, 1.618), 'sc': 2, 'prz_fib': 1.618},
            'shark':     {'AB_XA': (0.446, 0.618), 'BC_AB': (1.130, 1.618), 'CD_BC': (0.886, 1.130), 'AD_XA': (0.886, 1.130), 'sc': 2, 'prz_fib': 0.886},
            'cypher':    {'AB_XA': (0.382, 0.618), 'BC_AB': (1.130, 1.414), 'CD_BC': (0.786, 0.786), 'AD_XA': (0.786, 0.786), 'sc': 2, 'prz_fib': 0.786},
        }

        hi_idx, lo_idx = zigzag_swings_non_repaint(
            df['high'], df['low'],
            pct_thresh=zigzag_pct, confirmation_bars=confirmation_bars
        )
        swings = sorted(
            [{'idx': i, 'price': df['high'].iloc[i], 'type': 'H'} for i in hi_idx]
            + [{'idx': i, 'price': df['low'].iloc[i],  'type': 'L'} for i in lo_idx],
            key=lambda x: x['idx']
        )
        closes = df['close'].values

        for i in range(4, len(swings)):
            pts    = swings[i - 4:i + 1]
            types  = [p['type']  for p in pts]
            prices = [p['price'] for p in pts]
            idxs   = [p['idx']   for p in pts]
            Di     = idxs[4]
            if Di >= n:
                continue
            is_bear = (types == ['H', 'L', 'H', 'L', 'H'])
            is_bull = (types == ['L', 'H', 'L', 'H', 'L'])
            if not (is_bear or is_bull):
                continue
            X_p, A, B, C, D = prices
            XA = abs(A - X_p); AB = abs(B - A)
            BC = abs(C - B);   CD = abs(D - C); AD = abs(D - A)
            if any(v < 1e-9 for v in [XA, AB, BC, CD]):
                continue
            ab_xa = leg_ratio(XA, AB); bc_ab = leg_ratio(AB, BC)
            cd_bc = leg_ratio(BC, CD); ad_xa = leg_ratio(XA, AD)
            t = FIB_TOL
            for pn, r in PATS.items():
                la, ha = r['AB_XA']; lb, hb = r['BC_AB']
                lc, hc = r['CD_BC']; ld, hd = r['AD_XA']
                if (la - t <= ab_xa <= ha + t and lb - t <= bc_ab <= hb + t
                        and lc - t <= cd_bc <= hc + t and ld - t <= ad_xa <= hd + t):
                    acc = 1.0 - (
                        abs(ab_xa - (la + ha) / 2) / (ha - la + 0.01)
                        + abs(bc_ab - (lb + hb) / 2) / (hb - lb + 0.01)
                        + abs(cd_bc - (lc + hc) / 2) / (hc - lc + 0.01)
                    ) / 3
                    prz_c = (
                        X_p + (A - X_p) * r['prz_fib'] if is_bull
                        else X_p - (X_p - A) * r['prz_fib']
                    )
                    atr_d = max(df['atr'].iloc[Di], 1e-9)
                    df.at[Di, 'harmonic_pattern']   = 1.0
                    df.at[Di, 'harmonic_type']      = float(r['sc'])
                    df.at[Di, 'harmonic_direction'] = -1.0 if is_bear else 1.0
                    df.at[Di, 'harmonic_strength']  = max(0.0, min(1.0, acc))
                    df.at[Di, 'harmonic_prz']       = 1.0
                    df.at[Di, 'harmonic_prz_high']  = prz_c + atr_d * 0.6
                    df.at[Di, 'harmonic_prz_low']   = prz_c - atr_d * 0.6
                    df.at[Di, 'harmonic_valid']     = float(
                        (prz_c - atr_d * 0.6) <= closes[Di] <= (prz_c + atr_d * 0.6)
                    )
                    break
        return df

    # ══════════════════════════════════════════════════════════
    # ELLIOTT WAVE
    # ══════════════════════════════════════════════════════════
    def detect_elliott_wave(self, df, zigzag_pct=0.01, confirmation_bars=3):
        df = df.copy()
        n  = len(df)
        ew_cols = [
            'elliott_impulse', 'elliott_wave_num', 'elliott_corrective',
            'elliott_wave3', 'elliott_bullish', 'elliott_strength',
            'elliott_extended_w3', 'elliott_diagonal', 'elliott_valid',
            'elliott_w1_len', 'elliott_w3_len', 'elliott_w5_len'
        ]
        for col in ew_cols:
            df[col] = 0.0

        hi_idx, lo_idx = zigzag_swings_non_repaint(
            df['high'], df['low'],
            pct_thresh=zigzag_pct, confirmation_bars=confirmation_bars
        )
        swings = sorted(
            [{'idx': i, 'price': df['high'].iloc[i], 'type': 'H'} for i in hi_idx]
            + [{'idx': i, 'price': df['low'].iloc[i],  'type': 'L'} for i in lo_idx],
            key=lambda x: x['idx']
        )
        atr = df['atr'].values

        for i in range(4, len(swings)):
            pts    = swings[i - 4:i + 1]
            types  = [p['type']  for p in pts]
            prices = [p['price'] for p in pts]
            idxs   = [p['idx']   for p in pts]

            if types == ['L', 'H', 'L', 'H', 'L']:
                w0, w1, w2, w3, w4 = prices
                l1 = w1 - w0; l2 = w1 - w2; l3 = w3 - w2; l4 = w3 - w4
                l5 = abs(prices[4] - w4)
                valid = True
                if l1 <= 0 or l3 <= 0: valid = False
                if w2 <= w0:            valid = False
                if w4 <= w1:            valid = False
                if l3 < l1 * 0.85:     valid = False
                w2_r = l2 / (l1 + 1e-9)
                if not (0.236 <= w2_r <= 0.900): valid = False
                if valid:
                    w4_r = l4 / (l3 + 1e-9)
                    ext  = l3 > l1 * 1.618
                    alt  = not (w2_r > 0.5 and w4_r > 0.5)
                    w5ok = abs(l5 - l1) / (l1 + 1e-9) < 0.35
                    qb   = 1.0 + (0.25 if ext else 0) + (0.15 if alt else 0) + (0.10 if w5ok else 0)
                    ci   = idxs[-1]
                    if ci < n:
                        ai = max(atr[ci], 1e-9)
                        df.at[ci, 'elliott_impulse']  = 1.0
                        df.at[ci, 'elliott_wave_num'] = 5.0
                        df.at[ci, 'elliott_bullish']  = 1.0
                        df.at[ci, 'elliott_valid']    = 1.0
                        df.at[ci, 'elliott_strength'] = min(1.0, l3 / (ai * 5) * qb)
                        df.at[ci, 'elliott_w1_len']   = l1 / ai
                        df.at[ci, 'elliott_w3_len']   = l3 / ai
                        df.at[ci, 'elliott_w5_len']   = l5 / ai
                    w3i = idxs[3]
                    if w3i < n:
                        aw = max(atr[w3i], 1e-9)
                        df.at[w3i, 'elliott_wave3']       = 1.0
                        df.at[w3i, 'elliott_impulse']     = 1.0
                        df.at[w3i, 'elliott_wave_num']    = 3.0
                        df.at[w3i, 'elliott_bullish']     = 1.0
                        df.at[w3i, 'elliott_valid']       = 1.0
                        df.at[w3i, 'elliott_strength']    = min(1.0, l3 / (aw * 3) * qb)
                        df.at[w3i, 'elliott_extended_w3'] = float(ext)

            elif types == ['H', 'L', 'H', 'L', 'H']:
                w0, w1, w2, w3, w4 = prices
                l1 = w0 - w1; l2 = w2 - w1; l3 = w2 - w3; l4 = w4 - w3
                l5 = abs(prices[4] - w4)
                valid = True
                if l1 <= 0 or l3 <= 0: valid = False
                if w2 >= w0:            valid = False
                if w4 >= w1:            valid = False
                if l3 < l1 * 0.85:     valid = False
                w2_r = l2 / (l1 + 1e-9)
                if not (0.236 <= w2_r <= 0.900): valid = False
                if valid:
                    w4_r = l4 / (l3 + 1e-9)
                    ext  = l3 > l1 * 1.618
                    alt  = not (w2_r > 0.5 and w4_r > 0.5)
                    w5ok = abs(l5 - l1) / (l1 + 1e-9) < 0.35
                    qb   = 1.0 + (0.25 if ext else 0) + (0.15 if alt else 0) + (0.10 if w5ok else 0)
                    ci   = idxs[-1]
                    if ci < n:
                        ai = max(atr[ci], 1e-9)
                        df.at[ci, 'elliott_impulse']  = 1.0
                        df.at[ci, 'elliott_wave_num'] = 5.0
                        df.at[ci, 'elliott_bullish']  = 0.0
                        df.at[ci, 'elliott_valid']    = 1.0
                        df.at[ci, 'elliott_strength'] = min(1.0, l3 / (ai * 5) * qb)
                        df.at[ci, 'elliott_w1_len']   = l1 / ai
                        df.at[ci, 'elliott_w3_len']   = l3 / ai
                        df.at[ci, 'elliott_w5_len']   = l5 / ai
                    w3i = idxs[3]
                    if w3i < n:
                        aw = max(atr[w3i], 1e-9)
                        df.at[w3i, 'elliott_wave3']       = 1.0
                        df.at[w3i, 'elliott_impulse']     = 1.0
                        df.at[w3i, 'elliott_wave_num']    = 3.0
                        df.at[w3i, 'elliott_bullish']     = 0.0
                        df.at[w3i, 'elliott_valid']       = 1.0
                        df.at[w3i, 'elliott_strength']    = min(1.0, l3 / (aw * 3) * qb)
                        df.at[w3i, 'elliott_extended_w3'] = float(ext)

            if types in [['L', 'H', 'L', 'H', 'L'], ['H', 'L', 'H', 'L', 'H']]:
                legs = [abs(prices[j + 1] - prices[j]) for j in range(4)]
                if all(legs[j] < legs[j - 1] for j in range(1, 4)):
                    ci = idxs[-1]
                    if ci < n:
                        df.at[ci, 'elliott_diagonal'] = 1.0
                        df.at[ci, 'elliott_bullish']  = float(types == ['L', 'H', 'L', 'H', 'L'])
                        df.at[ci, 'elliott_valid']    = 1.0

        for i in range(2, len(swings)):
            pts    = swings[i - 2:i + 1]
            types  = [p['type']  for p in pts]
            prices = [p['price'] for p in pts]
            ci     = pts[-1]['idx']
            if ci >= n:
                continue
            if types == ['H', 'L', 'H']:
                A, B, C = prices; lA = A - B; lB = C - B
                if lA > 0 and lB > 0 and 0.382 <= lB / (lA + 1e-9) <= 1.618:
                    df.at[ci, 'elliott_corrective'] = 1.0
                    df.at[ci, 'elliott_bullish']    = 1.0
                    df.at[ci, 'elliott_valid']      = 1.0
            elif types == ['L', 'H', 'L']:
                A, B, C = prices; lA = B - A; lB = B - C
                if lA > 0 and lB > 0 and 0.382 <= lB / (lA + 1e-9) <= 1.618:
                    df.at[ci, 'elliott_corrective'] = 1.0
                    df.at[ci, 'elliott_bullish']    = 0.0
                    df.at[ci, 'elliott_valid']      = 1.0
        return df

    # ══════════════════════════════════════════════════════════
    # PATTERN META
    # ══════════════════════════════════════════════════════════
    def add_pattern_meta(self, df, max_age=20):
        df = df.copy()

        def _s(col, w=1.0, vcol=None):
            base = df[col].fillna(0) if col in df.columns else pd.Series(0.0, index=df.index)
            if vcol:
                vv = df[vcol].fillna(0) if vcol in df.columns else pd.Series(1.0, index=df.index)
                return base * w * vv
            return base * w

        pcols = [
            'hs_pattern', 'double_pattern', 'triple_pattern', 'triangle_pattern',
            'flag_pattern', 'wedge_pattern', 'channel_pattern',
            'cup_handle_pattern', 'rectangle_pattern',
            'harmonic_pattern', 'elliott_impulse', 'elliott_corrective'
        ]
        df['pattern_count'] = df[pcols].clip(0, 1).sum(axis=1)

        ew3  = df['elliott_wave3'].clip(0, 1)       if 'elliott_wave3'       in df.columns else pd.Series(0.0, index=df.index)
        ewx3 = df['elliott_extended_w3'].clip(0, 1) if 'elliott_extended_w3' in df.columns else pd.Series(0.0, index=df.index)

        df['pattern_cluster'] = (
            _s('harmonic_pattern') + _s('hs_pattern') + _s('double_pattern')
            + _s('triple_pattern') + ew3 + ewx3
        )
        d_mean = df[[c for c in ['hs_direction', 'double_direction', 'triple_direction',
                                  'harmonic_direction'] if c in df.columns]].mean(axis=1).abs()
        df['cluster_boost'] = (df['pattern_cluster'] * (1.0 + d_mean)).fillna(df['pattern_cluster'])
        df['strong_cluster'] = (df['cluster_boost'] >= 2.0).astype(float)

        df['pattern_quality'] = (
            _s('harmonic_strength',  3.0, 'harmonic_valid')
            + _s('hs_strength',      2.5, 'hs_valid')
            + _s('elliott_strength', 2.2, 'elliott_valid')
            + _s('triple_strength',  2.0, 'triple_valid')
            + _s('cup_handle_strength', 1.8, 'cup_handle_valid')
            + _s('double_strength',  1.5, 'double_valid')
            + _s('triangle_strength',1.3)
            + _s('flag_strength',    1.2)
            + _s('wedge_strength',   1.0)
            + _s('channel_strength', 0.8)
            + _s('rectangle_strength', 0.6)
        )
        qmax = df['pattern_quality'].max()
        df['pattern_quality'] = df['pattern_quality'] / (qmax + 1e-9)

        vcols = [c for c in ['hs_valid', 'double_valid', 'triple_valid',
                              'cup_handle_valid', 'harmonic_valid', 'elliott_valid']
                 if c in df.columns]
        df['pattern_valid'] = (
            (df[vcols].sum(axis=1) > 0).astype(float) if vcols
            else pd.Series(0.0, index=df.index)
        )

        dcols = [c for c in ['hs_direction', 'double_direction', 'triple_direction',
                              'triangle_direction', 'harmonic_direction', 'elliott_bullish']
                 if c in df.columns]
        df['direction_consensus'] = df[dcols].mean(axis=1) if dcols else 0.0

        df['bars_since_pattern'] = df['pattern_age']
        df['pattern_freshness']  = 1.0 / (df['bars_since_pattern'] + 1)
        df['pattern_expired']    = (df['pattern_age'] > max_age).astype(float)

        df['pattern_htf_aligned'] = (
            ((df['direction_consensus'] > 0) & (df['htf_trend'] > 0))
            | ((df['direction_consensus'] < 0) & (df['htf_trend'] < 0))
        ).astype(float)

        if 'pattern_volume_confirm' in df.columns:
            df['pattern_with_volume'] = (
                (df['pattern_count'] >= 1) & (df['pattern_volume_confirm'] == 1)
            ).astype(float)
        else:
            df['pattern_with_volume'] = 0.0

        df['pattern_score'] = (
            df['pattern_quality'] * df['pattern_freshness']
            * (1.0 + df['pattern_htf_aligned'] * 0.6)
            * (1.0 + df['pattern_with_volume'] * 0.4)
            * (1.0 + df['pattern_valid'] * 0.5)
        )

        # ★ v7.0: multi-confirmation score
        df['multi_confirm_score'] = (
            (df['pattern_valid'].astype(float)) +
            (df['pattern_htf_aligned'].astype(float)) +
            ((df['adx'] > 20).astype(float) if 'adx' in df.columns else 0) +
            (df['vol_surge'].astype(float) if 'vol_surge' in df.columns else 0)
        )

        if 'harmonic_prz_high' in df.columns:
            prz_h = df['harmonic_prz_high'].shift(1).fillna(0.0)
            prz_l = df['harmonic_prz_low'].shift(1).fillna(0.0)
            df['in_prz']      = ((df['close'] >= prz_l) & (df['close'] <= prz_h) & (prz_h > 0)).astype(float)
            df['dist_to_prz'] = (df['close'] - (prz_h + prz_l) / 2).abs() / (df['atr'] + 1e-9)
        else:
            df['in_prz']      = 0.0
            df['dist_to_prz'] = 0.0

        return df

    def create_features(self, df, config, shift=True):
        w  = config.get('pattern_window',    20)
        hr = config.get('htf_resample',      None)
        zp = config.get('zigzag_pct',        0.01)
        ma = config.get('pattern_max_age',   20)
        cb = config.get('confirmation_bars',  3)

        df = df.reset_index(drop=True).copy()
        df = self.add_base_features(df)
        df = self.add_htf_trend(df, htf_resample=hr)
        df = self.add_regime_features(df)
        df = self.add_order_flow(df)
        df = self.detect_classic_patterns(df, window=w, zigzag_pct=zp,
                                           max_age=ma, confirmation_bars=cb)
        df = self.detect_harmonic_patterns(df, zigzag_pct=zp, confirmation_bars=cb)
        df = self.detect_elliott_wave(df, zigzag_pct=zp, confirmation_bars=cb)
        df = self.add_pattern_meta(df, max_age=ma)

        lag_cols = ['rsi', 'macd_hist', 'mfi', 'bb_pctb', 'volatility',
                    'pattern_score', 'direction_consensus', 'stoch_rsi_k',
                    'cvd_norm', 'obv_slope_norm', 'rsi_divergence',
                    'adx', 'trend_strength', 'vol_ratio']
        for col in lag_cols:
            if col in df.columns:
                df[f'{col}_lag1'] = df[col].shift(1)
                df[f'{col}_lag2'] = df[col].shift(2)

        excl = {'datetime', 'open', 'high', 'low', 'close', 'volume', 'hour', 'dow'}
        if shift:
            for col in [c for c in df.columns if c not in excl]:
                df[col] = df[col].shift(1)

        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)
        warmup = max(10, min(60, len(df) // 10))
        df = df.iloc[warmup:].copy()

        thresh = int(len(df) * 0.3)
        df = df.dropna(axis=1, thresh=thresh)
        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        excl_f = {'datetime', 'open', 'high', 'low', 'close', 'volume',
                  'future_close', 'future_high', 'future_low',
                  'pct_change', 'up_potential', 'down_potential', 'target'}
        self.feature_columns = [c for c in df.columns if c not in excl_f]
        logging.info(f"Features: {len(self.feature_columns)}")
        return df

    # ★ v7.0: create_target محسَّن — متوازن بين Buy وSell
    def create_target(self, df, horizon=5, min_return=0.003,
                      atr_target_mult=1.5, _recursion_depth=0):
        """
        v7.0: Target متوازن:
        → 1 = BUY signal (up_potential > threshold)
        → 0 = SELL signal (down_potential > threshold)
        → -1 = لا إشارة (محذوف من التدريب)
        يحل مشكلة الـ SELL bias
        """
        if _recursion_depth > 3:
            logging.warning("create_target: وصلنا للحد الأقصى من المحاولات — نستخدم median split")
            df = df.copy()
            df['future_close'] = df['close'].shift(-horizon)
            df['pct_change']   = (df['future_close'] - df['close']) / (df['close'] + 1e-9)
            df['target'] = (df['pct_change'] > df['pct_change'].median()).astype(int)
            df.dropna(subset=['future_close'], inplace=True)
            return df

        df = df.copy()

        df['future_high'] = df['high'].shift(-1).rolling(horizon, min_periods=1).max().shift(-(horizon - 1))
        df['future_low']  = df['low'].shift(-1).rolling(horizon, min_periods=1).min().shift(-(horizon - 1))

        df['up_potential']   = (df['future_high'] - df['close']) / (df['atr'] + 1e-9)
        df['down_potential'] = (df['close'] - df['future_low'])  / (df['atr'] + 1e-9)

        fee_in_atr     = (TAKER_FEE * 2) / (df['atr_pct'].replace(0, np.nan).fillna(0.01))
        effective_mult = max(atr_target_mult, float(fee_in_atr.median() + atr_target_mult))
        effective_mult = min(effective_mult, atr_target_mult * 2)

        buy_signal  = (df['up_potential']   > effective_mult)
        sell_signal = (df['down_potential'] > effective_mult)

        # لو الاثنين triggered — خذ الأقوى
        both = buy_signal & sell_signal
        df['target'] = np.where(
            both,
            np.where(df['up_potential'] >= df['down_potential'], 1, 0),
            np.where(buy_signal, 1, np.where(sell_signal, 0, -1))
        )

        # احذف الصفوف بدون إشارة واضحة
        df_valid = df[df['target'] >= 0].copy()
        df_valid.dropna(subset=['future_high', 'future_low'], inplace=True)

        if len(df_valid) == 0:
            logging.error("create_target: لا يوجد بيانات بعد الفلترة!")
            return df.dropna(subset=['future_high', 'future_low'])

        buy_ratio  = (df_valid['target'] == 1).mean()
        sell_ratio = (df_valid['target'] == 0).mean()

        logging.info(
            f"v7.0 Target | {len(df_valid):,} صف | "
            f"Buy:{buy_ratio*100:.1f}% Sell:{sell_ratio*100:.1f}% | "
            f"atr_mult={effective_mult:.2f}"
        )

        # تحقق من التوازن
        if buy_ratio < 0.20:
            logging.warning(f"Target: Buy قليل جداً ({buy_ratio:.1%}) — نخفف المعامل")
            return self.create_target(df, horizon, min_return,
                                      atr_target_mult * 0.75, _recursion_depth + 1)
        elif buy_ratio > 0.80:
            logging.warning(f"Target: Sell قليل جداً ({1-buy_ratio:.1%}) — نرفع المعامل")
            return self.create_target(df, horizon, min_return,
                                      atr_target_mult * 1.25, _recursion_depth + 1)

        # تحذير لو مازال غير متوازن
        if abs(buy_ratio - 0.5) > 0.25:
            logging.warning(
                f"⚠️ Target غير متوازن (Buy={buy_ratio:.1%}) — "
                f"قد يوجد bias في الموديل"
            )

        return df_valid

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("scaler not fitted")
        if isinstance(X, pd.DataFrame):
            missing = [c for c in self.feature_columns if c not in X.columns]
            for c in missing:
                X[c] = 0.0
            X = X[self.feature_columns]
        return self.scaler.transform(X)

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)


# ══════════════════════════════════════════════════════════════
# XGBoost Builder
# ══════════════════════════════════════════════════════════════
def _build_xgb_v7(spw, n_est=800):
    return xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=n_est,
        max_depth=4,
        learning_rate=0.02,
        subsample=0.70,
        colsample_bytree=0.55,
        reg_alpha=1.0,
        reg_lambda=3.0,
        min_child_weight=12,       # ★ v7.0: 10→12 أقل overfitting
        gamma=0.3,
        scale_pos_weight=spw,
        random_state=42,
        tree_method='hist',
        verbosity=0,
        use_label_encoder=False,
        eval_metric='auc',
        early_stopping_rounds=50
    )


def _build_lgb_v7(spw):
    """LightGBM — رابع موديل في الـ ensemble"""
    if not HAS_LGB:
        return None
    return lgb.LGBMClassifier(
        n_estimators=600,
        max_depth=4,
        learning_rate=0.02,
        subsample=0.70,
        colsample_bytree=0.55,
        reg_alpha=1.0,
        reg_lambda=3.0,
        min_child_samples=20,
        scale_pos_weight=spw,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )


# ══════════════════════════════════════════════════════════════
# ★ v7.0: Ensemble مع Dynamic Weights
# ══════════════════════════════════════════════════════════════
def train_ensemble_v7(X_train_s, y_train, X_val_s, y_val, spw):
    """
    v7.0: Dynamic weights بناءً على AUC الفعلي لكل موديل
    + LightGBM كرابع موديل
    """
    models   = {}
    val_aucs = {}
    val_preds = {}

    def _safe_auc(y_true, y_pred):
        if len(y_true.unique()) < 2:
            return 0.5
        try:
            return roc_auc_score(y_true, y_pred)
        except Exception:
            return 0.5

    # XGBoost
    xgb_mdl = _build_xgb_v7(spw)
    xgb_mdl.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)
    xgb_pred = xgb_mdl.predict_proba(X_val_s)[:, 1]
    models['xgb']    = xgb_mdl
    val_preds['xgb'] = xgb_pred
    val_aucs['xgb']  = _safe_auc(y_val, xgb_pred)

    # RandomForest
    rf_mdl = RandomForestClassifier(
        n_estimators=300, max_depth=6,
        min_samples_leaf=20, min_samples_split=30,
        max_features='sqrt', random_state=42,
        class_weight='balanced', n_jobs=-1
    )
    rf_mdl.fit(X_train_s, y_train)
    rf_pred = rf_mdl.predict_proba(X_val_s)[:, 1]
    models['rf']    = rf_mdl
    val_preds['rf'] = rf_pred
    val_aucs['rf']  = _safe_auc(y_val, rf_pred)

    # Logistic Regression
    lr_mdl = LogisticRegression(
        C=0.05, random_state=42, class_weight='balanced',
        max_iter=2000, solver='lbfgs'
    )
    lr_mdl.fit(X_train_s, y_train)
    lr_pred = lr_mdl.predict_proba(X_val_s)[:, 1]
    models['lr']    = lr_mdl
    val_preds['lr'] = lr_pred
    val_aucs['lr']  = _safe_auc(y_val, lr_pred)

    # LightGBM (اختياري)
    if HAS_LGB:
        lgb_mdl = _build_lgb_v7(spw)
        lgb_mdl.fit(
            X_train_s, y_train,
            eval_set=[(X_val_s, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )
        lgb_pred = lgb_mdl.predict_proba(X_val_s)[:, 1]
        models['lgb']    = lgb_mdl
        val_preds['lgb'] = lgb_pred
        val_aucs['lgb']  = _safe_auc(y_val, lgb_pred)

    # ★ Dynamic Weights: AUC - 0.5 (بيمثل القدرة الفعلية فوق العشوائية)
    raw_weights = {k: max(0.0, v - 0.5) for k, v in val_aucs.items()}
    total_w     = sum(raw_weights.values())

    if total_w < 0.01:
        # fallback: weights متساوية
        n_models = len(models)
        dynamic_weights = {k: 1.0 / n_models for k in models}
    else:
        dynamic_weights = {k: v / total_w for k, v in raw_weights.items()}

    logging.info(f"Val AUCs  : {', '.join(f'{k}={v:.3f}' for k,v in val_aucs.items())}")
    logging.info(f"Dyn Weights: {', '.join(f'{k}={v:.2f}' for k,v in dynamic_weights.items())}")

    ensemble_pred = sum(
        val_preds[k] * dynamic_weights[k] for k in models
    )

    return models, ensemble_pred, dynamic_weights


class EnsemblePredictor:
    """
    v7.0: يدعم dynamic weights وعدد موديلات مرن (3 أو 4)
    """
    def __init__(self, models, weights=None):
        self.models  = models
        n = len(models)
        self.weights = weights or {k: 1.0 / n for k in models}

    def predict_proba(self, X):
        preds = np.zeros(len(X))
        for name, mdl in self.models.items():
            w = self.weights.get(name, 0.0)
            preds += mdl.predict_proba(X)[:, 1] * w
        return np.column_stack([1 - preds, preds])

    def predict(self, X, threshold=0.50):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


# ══════════════════════════════════════════════════════════════
# Adaptive Threshold
# ══════════════════════════════════════════════════════════════
def get_adaptive_threshold(cfg, fold_trades, tf, use_fold=True):
    min_t = MIN_TRADES_PER_FOLD.get(tf, 8)
    if use_fold:
        base_th = cfg.get('fold_buy_threshold', 0.50)
        base_st = cfg.get('fold_sell_threshold', 0.50)
    else:
        base_th = cfg.get('buy_threshold', 0.52)
        base_st = cfg.get('sell_threshold', 0.48)

    if fold_trades < min_t:
        ratio     = fold_trades / max(min_t, 1)
        reduction = (1 - ratio) * 0.04
        new_th    = max(0.50, base_th - reduction)
        new_st    = min(0.50, base_st + reduction)
        return new_th, new_st

    return base_th, base_st


# ══════════════════════════════════════════════════════════════
# ★ v7.0: Backtester المُحسَّن مع Circuit Breaker + Multi-confirm
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self,
                 initial_capital=10000,
                 spread_bps=SPREAD_BPS,
                 risk_per_trade=0.006,
                 max_positions=1,
                 max_position_pct=0.05,
                 max_drawdown_pct=0.30,
                 min_pattern_quality=0.08,
                 require_valid=False,
                 confidence_threshold=0.50,
                 atr_sl_mult=ATR_SL_MULT,
                 atr_tp_mult=ATR_TP_MULT,
                 is_walk_forward=False,
                 # ★ v7.0: جديد
                 min_pattern_score=0.10,
                 min_multi_confirm=1,
                 use_htf_filter=True,
                 time_filter_hours=None):
        self.initial_capital      = initial_capital
        self.spread               = spread_bps / 10000
        self.taker_fee            = TAKER_FEE
        self.slippage             = 0.0002
        self.risk_per_trade       = risk_per_trade
        self.max_positions        = max_positions
        self.max_position_pct     = max_position_pct
        self.max_drawdown_pct     = max_drawdown_pct
        self.min_pattern_quality  = min_pattern_quality
        self.require_valid        = require_valid
        self.confidence_threshold = confidence_threshold
        self.atr_sl_mult          = atr_sl_mult
        self.atr_tp_mult          = atr_tp_mult
        self.is_walk_forward      = is_walk_forward
        self.min_pattern_score    = min_pattern_score    # ★ v7.0
        self.min_multi_confirm    = min_multi_confirm    # ★ v7.0
        self.use_htf_filter       = use_htf_filter       # ★ v7.0
        self.time_filter_hours    = time_filter_hours    # ★ v7.0

    def _exec(self, p, buy):
        if buy:
            return p * (1 + self.spread) * (1 + self.slippage) * (1 + self.taker_fee)
        return p * (1 - self.spread) * (1 - self.slippage) * (1 - self.taker_fee)

    def run(self, df, model, fe, feature_columns, cfg, atr_sl=None, atr_tp=None,
            use_fold=False, tf='4h'):
        capital   = self.initial_capital
        positions = []
        trades    = []
        equity    = [capital]

        missing = [c for c in feature_columns + ['close', 'atr'] if c not in df.columns]
        if missing:
            logging.error(f"Missing columns: {missing}")
            return {}

        bth = cfg.get('fold_buy_threshold', 0.50) if use_fold else cfg.get('buy_threshold', 0.52)
        sth = cfg.get('fold_sell_threshold', 0.50) if use_fold else cfg.get('sell_threshold', 0.48)

        sl_mult = self.atr_sl_mult if atr_sl is None else atr_sl
        tp_mult = self.atr_tp_mult if atr_tp is None else atr_tp

        if self.is_walk_forward:
            eff_min_pq      = 0.03
            eff_adx         = 3.0
            eff_min_ps      = 0.05
            eff_min_confirm = 0
        else:
            eff_min_pq      = self.min_pattern_quality
            eff_adx         = 8.0
            eff_min_ps      = self.min_pattern_score
            eff_min_confirm = self.min_multi_confirm

        proba  = model.predict_proba(fe.transform(df[feature_columns].copy()))[:, 1]
        closes = df['close'].values
        atrs   = df['atr'].values
        hours  = df['datetime'].dt.hour.values if 'datetime' in df.columns else np.zeros(len(df))

        pq   = df['pattern_quality'].values   if 'pattern_quality'   in df.columns else np.full(len(df), 1.0)
        ps   = df['pattern_score'].values     if 'pattern_score'     in df.columns else np.full(len(df), 1.0)
        pv   = df['pattern_valid'].values     if 'pattern_valid'     in df.columns else np.ones(len(df))
        htf  = df['htf_trend'].values         if 'htf_trend'         in df.columns else np.zeros(len(df))
        adxv = df['adx'].values               if 'adx'               in df.columns else np.full(len(df), 20.0)
        volr = df['vol_regime_btc'].values    if 'vol_regime_btc'    in df.columns else np.full(len(df), 1)
        kcsq = df['kc_squeeze'].values        if 'kc_squeeze'        in df.columns else np.zeros(len(df))
        mcs  = df['multi_confirm_score'].values if 'multi_confirm_score' in df.columns else np.ones(len(df))
        bull = df['bull_market'].values       if 'bull_market'       in df.columns else np.full(len(df), 0.5)

        peak_capital      = capital
        # ★ v7.0: Circuit Breaker
        consecutive_losses = 0
        cooling_off_until  = -1

        for i in range(len(df)):
            p, a, pr = closes[i], max(float(atrs[i]), 1e-9), proba[i]
            h_now = int(hours[i])

            # Adaptive SL/TP
            regime = volr[i]
            if regime == 0:
                curr_sl_m = sl_mult * 0.90; curr_tp_m = tp_mult * 1.20
            elif regime == 2:
                curr_sl_m = sl_mult * 1.30; curr_tp_m = tp_mult * 1.50
            else:
                curr_sl_m = sl_mult;        curr_tp_m = tp_mult

            # ★ v7.0: Regime-aware thresholds
            if adxv[i] > 25:         # Trending market
                adj_bth = max(0.48, bth - 0.02)
                adj_sth = min(0.52, sth + 0.02)
            elif adxv[i] < 15:       # Ranging market
                adj_bth = min(0.60, bth + 0.03)
                adj_sth = max(0.40, sth - 0.03)
            else:
                adj_bth = bth; adj_sth = sth

            curr_sl_dist = curr_sl_m * a
            curr_tp_dist = curr_tp_m * a

            peak_capital = max(peak_capital, capital)
            curr_dd = (capital - peak_capital) / (peak_capital + 1e-9)

            # ★ v7.0: Circuit Breaker — لو في cooling period
            in_cooling = (i <= cooling_off_until)

            if curr_dd < -self.max_drawdown_pct or in_cooling:
                sig = "HOLD"
            else:
                raw = "BUY" if pr > adj_bth else ("SELL" if pr < adj_sth else "HOLD")

                # فلاتر متعددة
                if raw != "HOLD":
                    # pattern quality
                    if pq[i] < eff_min_pq:
                        raw = "HOLD"
                    # pattern score
                    elif ps[i] < eff_min_ps:
                        raw = "HOLD"
                    # valid pattern
                    elif self.require_valid and pv[i] < 1.0:
                        raw = "HOLD"
                    # ADX filter
                    elif adxv[i] < eff_adx:
                        raw = "HOLD"
                    # ★ v7.0: multi-confirmation filter (فقط في non-WF)
                    elif not self.is_walk_forward and mcs[i] < eff_min_confirm:
                        raw = "HOLD"
                    # HTF filter (مع احترام use_htf_filter)
                    elif (not self.is_walk_forward
                          and self.use_htf_filter
                          and raw == "BUY"  and htf[i] < 0):
                        raw = "HOLD"
                    elif (not self.is_walk_forward
                          and self.use_htf_filter
                          and raw == "SELL" and htf[i] > 0):
                        raw = "HOLD"
                    # squeeze filter
                    elif (not self.is_walk_forward
                          and raw != "HOLD"
                          and kcsq[i] == 1
                          and volr[i] == 0
                          and adxv[i] < 8):
                        raw = "HOLD"
                    # ★ v7.0: time filter
                    elif (self.time_filter_hours is not None
                          and not self.is_walk_forward):
                        h_min, h_max = self.time_filter_hours
                        if not (h_min <= h_now <= h_max):
                            raw = "HOLD"

                sig = raw

            # SL/TP Management مع Trailing Stop
            to_close = []
            for pos in positions[:]:
                if pos['type'] == 'buy':
                    if p <= pos['sl']:
                        pnl = pos['units'] * (pos['sl'] - pos['entry'])
                        capital += pnl
                        trades.append({'profit': pnl, 'exit': 'sl', 'direction': 1,
                                       'entry': pos['entry'], 'exit_price': pos['sl']})
                        to_close.append(pos)
                    elif p >= pos['tp']:
                        pnl = pos['units'] * (pos['tp'] - pos['entry'])
                        capital += pnl
                        trades.append({'profit': pnl, 'exit': 'tp', 'direction': 1,
                                       'entry': pos['entry'], 'exit_price': pos['tp']})
                        to_close.append(pos)
                    elif (p >= pos.get('tp1', pos['tp'])
                          and not pos.get('tp1_hit', False)):
                        pos['tp1_hit'] = True
                        # ★ v7.0: trailing stop بدل breakeven مباشرة
                        pos['sl'] = pos['entry'] + (a * 0.5)  # trailing stop: breakeven + 0.5 ATR
                else:
                    if p >= pos['sl']:
                        pnl = pos['units'] * (pos['entry'] - pos['sl'])
                        capital += pnl
                        trades.append({'profit': pnl, 'exit': 'sl', 'direction': -1,
                                       'entry': pos['entry'], 'exit_price': pos['sl']})
                        to_close.append(pos)
                    elif p <= pos['tp']:
                        pnl = pos['units'] * (pos['entry'] - pos['tp'])
                        capital += pnl
                        trades.append({'profit': pnl, 'exit': 'tp', 'direction': -1,
                                       'entry': pos['entry'], 'exit_price': pos['tp']})
                        to_close.append(pos)
                    elif (p <= pos.get('tp1', pos['tp'])
                          and not pos.get('tp1_hit', False)):
                        pos['tp1_hit'] = True
                        pos['sl'] = pos['entry'] - (a * 0.5)  # ★ v7.0: trailing

            for pos in to_close:
                if pos in positions:
                    positions.remove(pos)
                    # ★ v7.0: Circuit Breaker logic
                    last_trade = trades[-1] if trades else None
                    if last_trade:
                        if last_trade['profit'] <= 0:
                            consecutive_losses += 1
                            if consecutive_losses >= MAX_CONSEC_LOSSES:
                                cooling_off_until  = i + COOLING_BARS
                                consecutive_losses = 0
                                logging.debug(
                                    f"🔴 Circuit Breaker: cooling until bar {cooling_off_until}"
                                )
                        else:
                            consecutive_losses = 0

            # Open New Position
            if sig == "BUY" and len(positions) < self.max_positions and a > 0:
                # ★ v7.0: وزن الإشارة حسب bull_market
                risk_adj  = self.risk_per_trade * (1.2 if bull[i] > 0.5 else 0.8)
                ra        = capital * risk_adj
                units     = ra / curr_sl_dist if curr_sl_dist > 0 else 0
                if units > 0:
                    entry = self._exec(p, True)
                    mc    = capital * self.max_position_pct
                    cost  = min(units * entry, mc)
                    units = cost / entry if entry > 0 else 0
                    if cost > 0 and cost <= capital:
                        capital -= cost
                        tp1_p = entry + (curr_tp_dist * 0.40)
                        positions.append({
                            'type': 'buy', 'entry': entry, 'units': units,
                            'sl': entry - curr_sl_dist, 'tp': entry + curr_tp_dist,
                            'tp1': tp1_p, 'margin': cost, 'tp1_hit': False
                        })

            elif sig == "SELL" and len(positions) < self.max_positions and a > 0:
                risk_adj  = self.risk_per_trade * (0.8 if bull[i] > 0.5 else 1.2)
                ra        = capital * risk_adj
                units     = ra / curr_sl_dist if curr_sl_dist > 0 else 0
                if units > 0:
                    entry  = self._exec(p, False)
                    margin = min(ra, capital * self.max_position_pct)
                    if margin > 0 and margin <= capital:
                        capital -= margin
                        tp1_p = entry - (curr_tp_dist * 0.40)
                        positions.append({
                            'type': 'sell', 'entry': entry, 'units': units,
                            'sl': entry + curr_sl_dist, 'tp': entry - curr_tp_dist,
                            'tp1': tp1_p, 'margin': margin, 'tp1_hit': False
                        })

            # Open P&L
            ov = sum(
                (pos['units'] * (p - pos['entry']) if pos['type'] == 'buy'
                 else pos['units'] * (pos['entry'] - p))
                for pos in positions
            )
            equity.append(capital + ov)

        # إغلاق المتبقي
        for pos in positions:
            lp = closes[-1]
            if pos['type'] == 'buy':
                exit_p = self._exec(lp, False)
                pnl    = pos['units'] * (exit_p - pos['entry'])
            else:
                exit_p = self._exec(lp, True)
                pnl    = pos['units'] * (pos['entry'] - exit_p)
            capital += pnl
            trades.append({'profit': pnl, 'exit': 'eod',
                           'direction': 1 if pos['type'] == 'buy' else -1,
                           'entry': pos['entry'], 'exit_price': lp})

        if not trades:
            return {}

        wins = [t for t in trades if t['profit'] > 0]
        loss = [t for t in trades if t['profit'] <= 0]
        wr   = len(wins) / len(trades) * 100
        tp_s = sum(t['profit'] for t in wins)
        tl_s = abs(sum(t['profit'] for t in loss))
        aw   = tp_s / len(wins) if wins else 0
        al   = tl_s / len(loss) if loss else 0
        pf   = min(99.9, tp_s / tl_s) if tl_s > 0 else (99.9 if tp_s > 0 else 0)
        ret  = (capital - self.initial_capital) / self.initial_capital * 100
        eq_s = pd.Series(equity)
        dd   = float(((eq_s - eq_s.cummax()) / (eq_s.cummax() + 1e-9) * 100).min())

        trade_rets   = [t['profit'] / self.initial_capital for t in trades]
        sharpe_proxy = (
            (np.mean(trade_rets) / (np.std(trade_rets) + 1e-9)) * np.sqrt(252)
            if len(trade_rets) > 1 else 0
        )

        # ★ v7.0: direction balance
        buy_trades  = [t for t in trades if t['direction'] == 1]
        sell_trades = [t for t in trades if t['direction'] == -1]
        dir_balance = len(buy_trades) / max(len(trades), 1)

        return {
            'total_trades':     len(trades),
            'win_rate':         round(wr, 2),
            'profit_factor':    round(pf, 2),
            'total_return_pct': round(ret, 2),
            'max_drawdown_pct': round(dd, 2),
            'avg_win':          round(aw, 2),
            'avg_loss':         round(al, 2),
            'expectancy':       round((wr / 100 * aw) - ((1 - wr / 100) * al), 2) if al else 0,
            'sharpe_proxy':     round(sharpe_proxy, 3),
            'buy_trades':       len(buy_trades),
            'sell_trades':      len(sell_trades),
            'direction_balance': round(dir_balance, 3),   # ★ v7.0: 0=all sell, 1=all buy
            'trades':           trades
        }


# ══════════════════════════════════════════════════════════════
# ★ v7.0: Walk-Forward Expanding Window
# ══════════════════════════════════════════════════════════════
def walk_forward_expanding_v7(X, y, df_tgt, cfg, n_splits=6,
                               purge_pct=None, tf_name=''):
    if purge_pct is None:
        if tf_name in ('1d', '1W'):
            purge_pct = 0.06      # ★ v7.0: أكبر لمنع leakage
        elif tf_name in ('4h', '2h', '1h'):
            purge_pct = 0.05
        else:
            purge_pct = 0.03

    n          = len(X)
    purge_size = max(1, int(n * purge_pct))
    min_init   = min(200, max(n // 3, 10))
    init_train = max(int(n * 0.40), min_init)
    remaining  = n - init_train
    test_size  = max(int(remaining / n_splits), max(20, n // 15))

    min_pq_wf           = 0.03
    min_trades_for_score = 10

    use_htf_filter = cfg.get('use_htf_filter', True)

    logging.info(
        f"  WF v7.0: n={n} | init={init_train} | test={test_size} | "
        f"purge={purge_size} | Ensemble=Dynamic | HTF_filter={use_htf_filter}"
    )

    fold_results  = []
    best_ensemble = None
    best_fe       = None
    best_score    = -np.inf
    last_ensemble = None
    last_fef      = None

    for fold in range(n_splits):
        train_end  = init_train + fold * test_size
        test_start = train_end + purge_size
        test_end   = min(test_start + test_size, n)

        if test_start >= n or test_end <= test_start:
            break

        tei = list(range(test_start, test_end))
        if len(y.iloc[tei].unique()) < 2:
            resolved = False
            for ext_mult in [1, 2, 3]:
                ext_end = min(test_end + test_size * ext_mult, n)
                tei_ext = list(range(test_start, ext_end))
                if len(y.iloc[tei_ext].unique()) >= 2:
                    tei = tei_ext
                    logging.warning(f"  Fold {fold+1}: توسيع ×{ext_mult+1} → {len(tei)} عينة")
                    resolved = True
                    break
            if not resolved:
                logging.warning(f"  Fold {fold+1}: فئة واحدة — تخطي")
                continue

        tri = list(range(0, train_end))
        Xtr, Xte = X.iloc[tri], X.iloc[tei]
        ytr, yte = y.iloc[tri], y.iloc[tei]

        if len(ytr.unique()) < 2:
            mc  = 1 if (ytr == 0).all() else 0
            Xtr = pd.concat([Xtr, Xte.iloc[:1]])
            ytr = pd.concat([ytr, pd.Series([mc], index=[Xte.index[0]])])

        neg = (ytr == 0).sum()
        pos = (ytr == 1).sum()
        spw = neg / pos if pos > 0 else 1.0

        fef = PatternFeatureEngineer()
        fef.feature_columns = list(X.columns)
        fef.fit_scaler(Xtr)
        Xtrs = fef.transform(Xtr)
        Xtes = fef.transform(Xte)

        # ★ v7.0: train dynamic ensemble
        models_dict, ensemble_pred, dyn_weights = train_ensemble_v7(
            Xtrs, ytr, Xtes, yte, spw
        )
        ensemble = EnsemblePredictor(models_dict, weights=dyn_weights)

        yp_ens = (ensemble_pred >= 0.50).astype(int)
        acc    = accuracy_score(yte, yp_ens)
        pr     = precision_score(yte, yp_ens, zero_division=0)
        rc     = recall_score(yte, yp_ens,    zero_division=0)
        try:
            auc = roc_auc_score(yte, ensemble_pred) if len(yte.unique()) > 1 else 0.5
        except Exception:
            auc = 0.5

        df_fold = df_tgt.iloc[tei].copy()

        mbt = Backtester(
            require_valid=False,
            min_pattern_quality=min_pq_wf,
            min_pattern_score=0.03,
            min_multi_confirm=0,
            use_htf_filter=use_htf_filter,
            confidence_threshold=cfg.get('fold_buy_threshold', 0.50),
            atr_sl_mult=cfg.get('atr_sl_mult', ATR_SL_MULT),
            atr_tp_mult=cfg.get('atr_tp_mult', ATR_TP_MULT),
            is_walk_forward=True
        ).run(df_fold, ensemble, fef, fef.feature_columns, cfg,
              use_fold=True, tf=tf_name)

        fst = (str(df_tgt['datetime'].iloc[test_start])
               if 'datetime' in df_tgt.columns else '')
        fet = (str(df_tgt['datetime'].iloc[min(test_end - 1, len(df_tgt) - 1)])
               if 'datetime' in df_tgt.columns else '')

        logging.info(
            f"  Fold {fold+1}/{n_splits} ({fst[:10]}→{fet[:10]}) "
            f"Acc:{acc:.2%} AUC:{auc:.3f} Pr:{pr:.2%} Rc:{rc:.2%}"
        )

        t_count = 0
        if mbt:
            t_count = mbt.get('total_trades', 0)
            dir_b   = mbt.get('direction_balance', 0.5)
            logging.info(
                f"    BT | T:{t_count} WR:{mbt['win_rate']}% "
                f"PF:{mbt['profit_factor']} R:{mbt['total_return_pct']}% "
                f"DD:{mbt['max_drawdown_pct']}% Sharpe:{mbt.get('sharpe_proxy',0):.3f} "
                f"Buy/Sell balance:{dir_b:.2f}"
            )

        fold_results.append({
            'fold': fold + 1, 'train_size': train_end, 'test_size': len(tei),
            'purge_size': purge_size, 'period_start': fst, 'period_end': fet,
            'accuracy': round(acc, 4), 'precision': round(pr, 4),
            'recall': round(rc, 4), 'auc': round(auc, 4),
            'dynamic_weights': dyn_weights,
            'backtest': mbt
        })

        last_ensemble = ensemble
        last_fef      = fef

        if mbt and t_count >= min_trades_for_score:
            sharpe  = mbt.get('sharpe_proxy', 0)
            pf_cap  = min(mbt.get('profit_factor', 1.0), 2.5)
            t_conf  = min(t_count / 15.0, 1.0)
            sc = (auc * 0.35) + (pf_cap / 2.5 * 0.35) + (max(0, sharpe) / 3 * 0.30)
            sc = sc + acc * 0.10
            sc = sc * t_conf
        else:
            sc = auc * 0.5 + acc * 0.5

        if sc > best_score:
            best_score    = sc
            best_ensemble = ensemble
            best_fe       = fef

    if best_ensemble is None:
        if last_ensemble is not None:
            logging.warning("⚠️ Fallback: آخر fold مدرَّب")
            best_ensemble = last_ensemble
            best_fe       = last_fef
        else:
            raise ValueError("Walk-Forward فشل: كل الـ folds تم تخطيها")

    return fold_results, best_ensemble, best_fe


# ══════════════════════════════════════════════════════════════
# ★ v7.0: Simple Split للـ 1W (بيانات قليلة جداً)
# ══════════════════════════════════════════════════════════════
def simple_time_split_v7(X, y, df_tgt, cfg, train_ratio=0.70, tf_name='1W'):
    """
    بدل WF للفريمات التي لا تملك بيانات كافية
    70% train / 30% test — split واحد
    """
    n          = len(X)
    train_end  = int(n * train_ratio)
    test_start = train_end

    logging.info(f"  Simple Split v7.0: n={n} | train={train_end} | test={n-train_end}")

    Xtr, Xte = X.iloc[:train_end], X.iloc[test_start:]
    ytr, yte = y.iloc[:train_end], y.iloc[test_start:]

    if len(ytr.unique()) < 2 or len(yte.unique()) < 2:
        logging.warning("Simple Split: فئة واحدة في train أو test")
        return [], None, None

    neg = (ytr == 0).sum()
    pos = (ytr == 1).sum()
    spw = neg / pos if pos > 0 else 1.0

    fef = PatternFeatureEngineer()
    fef.feature_columns = list(X.columns)
    fef.fit_scaler(Xtr)
    Xtrs = fef.transform(Xtr)
    Xtes = fef.transform(Xte)

    models_dict, ensemble_pred, dyn_weights = train_ensemble_v7(
        Xtrs, ytr, Xtes, yte, spw
    )
    ensemble = EnsemblePredictor(models_dict, weights=dyn_weights)

    yp = (ensemble_pred >= 0.50).astype(int)
    acc = accuracy_score(yte, yp)
    try:
        auc = roc_auc_score(yte, ensemble_pred) if len(yte.unique()) > 1 else 0.5
    except Exception:
        auc = 0.5

    df_test = df_tgt.iloc[test_start:].copy()
    mbt = Backtester(
        require_valid=False, min_pattern_quality=0.03,
        min_pattern_score=0.03, use_htf_filter=False,
        is_walk_forward=True
    ).run(df_test, ensemble, fef, fef.feature_columns, cfg, use_fold=True, tf=tf_name)

    fst = str(df_tgt['datetime'].iloc[test_start]) if 'datetime' in df_tgt.columns else ''
    fet = str(df_tgt['datetime'].iloc[-1])          if 'datetime' in df_tgt.columns else ''

    logging.info(f"  Simple Split | Acc:{acc:.2%} AUC:{auc:.3f}")
    if mbt:
        logging.info(
            f"  BT | T:{mbt['total_trades']} WR:{mbt['win_rate']}% "
            f"PF:{mbt['profit_factor']} Sharpe:{mbt.get('sharpe_proxy',0):.3f}"
        )

    fold_results = [{
        'fold': 1, 'train_size': train_end, 'test_size': n - train_end,
        'purge_size': 0, 'period_start': fst, 'period_end': fet,
        'accuracy': round(acc, 4), 'precision': 0, 'recall': 0,
        'auc': round(auc, 4), 'dynamic_weights': dyn_weights, 'backtest': mbt
    }]

    return fold_results, ensemble, fef


# ══════════════════════════════════════════════════════════════
# Multi-Timeframe Decision Layer
# ══════════════════════════════════════════════════════════════
def multi_timeframe_decision(signals: dict) -> dict:
    if not signals:
        return {"final_signal": "NO TRADE", "confidence": 0.0, "reason": "لا إشارات"}

    TF_W = {"1W": 5, "1d": 4, "4h": 3.5, "2h": 3,
            "1h": 2.5, "45min": 1.5, "15min": 1, "5min": 0.5}

    dirs      = {tf: v.get('direction', 0)         for tf, v in signals.items()}
    probs     = {tf: v.get('prob', 0.5)            for tf, v in signals.items()}
    qualities = {tf: v.get('pattern_quality', 0.0) for tf, v in signals.items()}
    valids    = {tf: v.get('pattern_valid', 0.0)   for tf, v in signals.items()}
    htf_t     = {tf: v.get('htf_trend', 0)         for tf, v in signals.items()}

    wd = tw = 0.0
    for tf, d in dirs.items():
        w   = TF_W.get(tf, 1.0)
        wd += d * w * probs.get(tf, 0.5)
        tw += w
    if tw > 0:
        wd /= tw

    avg_q  = float(np.mean(list(qualities.values()))) if qualities else 0.0
    avg_p  = float(np.mean(list(probs.values())))     if probs    else 0.5
    avg_v  = float(np.mean(list(valids.values())))    if valids   else 0.0
    htf_ok = all(htf_t.get(tf, 0) * dirs.get(tf, 0) >= 0
                 for tf in signals if dirs.get(tf, 0) != 0)

    if avg_q < 0.05:    # ★ v7.0: خُفِّض من 0.08
        return {"final_signal": "NO TRADE", "confidence": 0.0,
                "reason": f"جودة منخفضة ({avg_q:.2f})"}

    if avg_p > 0.75 and avg_q > 0.45 and avg_v > 0.5:
        cl = "strong"
    elif avg_p > 0.60 and avg_q > 0.25:
        cl = "medium"
    else:
        cl = "weak"

    all_buy  = all(d > 0 for d in dirs.values() if d != 0)
    all_sell = all(d < 0 for d in dirs.values() if d != 0)
    maj_buy  = wd > 0.25
    maj_sell = wd < -0.25
    reason   = ["⚠️ HTF غير متوافق"] if not htf_ok else []

    if   all_buy  and htf_ok and cl == "strong":  sig = "STRONG BUY 🔥";  conf = min(0.95, avg_p); reason.append("كل الفريمات BUY")
    elif all_sell and htf_ok and cl == "strong":  sig = "STRONG SELL 🔥"; conf = min(0.95, avg_p); reason.append("كل الفريمات SELL")
    elif maj_buy  and htf_ok and cl != "weak":    sig = "BUY 📊";          conf = avg_p * 0.85
    elif maj_sell and htf_ok and cl != "weak":    sig = "SELL 📊";         conf = avg_p * 0.85
    elif maj_buy  and not htf_ok:                 sig = "WEAK BUY ⚠️";    conf = avg_p * 0.45
    elif maj_sell and not htf_ok:                 sig = "WEAK SELL ⚠️";   conf = avg_p * 0.45
    else:                                          sig = "NO TRADE ❌";     conf = 0.0; reason.append(f"تعارض (wd={wd:.2f})")

    return {
        "final_signal":       sig,
        "confidence":         round(conf, 3),
        "confidence_level":   cl,
        "weighted_direction": round(wd, 3),
        "avg_quality":        round(avg_q, 3),
        "avg_valid":          round(avg_v, 3),
        "avg_prob":           round(avg_p, 3),
        "htf_aligned":        htf_ok,
        "reason":             " | ".join(reason),
        "tf_signals":         signals
    }


def calculate_btc_targets(entry_price: float, direction: int = 1,
                           atr_value: float = None) -> dict:
    if atr_value is None or atr_value <= 0:
        atr_value = entry_price * 0.015

    sl_dist  = ATR_SL_MULT * atr_value
    tp1_dist = ATR_TP_MULT * atr_value * 0.40
    tp_dist  = ATR_TP_MULT * atr_value
    rr       = ATR_TP_MULT / ATR_SL_MULT

    if direction == 1:
        return {
            "direction": "BUY", "entry": round(entry_price, 2),
            "sl": round(entry_price - sl_dist, 2),
            "tp1": round(entry_price + tp1_dist, 2),
            "tp_full": round(entry_price + tp_dist, 2),
            "sl_dist_usd": round(sl_dist, 2), "tp_dist_usd": round(tp_dist, 2),
            "sl_pct": round(sl_dist / entry_price * 100, 3),
            "tp_pct": round(tp_dist / entry_price * 100, 3),
            "rr": round(rr, 2),
            "fee_both_legs": round(TAKER_FEE * 2 * 100, 4),
            "atr_used": round(atr_value, 2),
        }
    else:
        return {
            "direction": "SELL", "entry": round(entry_price, 2),
            "sl": round(entry_price + sl_dist, 2),
            "tp1": round(entry_price - tp1_dist, 2),
            "tp_full": round(entry_price - tp_dist, 2),
            "sl_dist_usd": round(sl_dist, 2), "tp_dist_usd": round(tp_dist, 2),
            "sl_pct": round(sl_dist / entry_price * 100, 3),
            "tp_pct": round(tp_dist / entry_price * 100, 3),
            "rr": round(rr, 2),
            "fee_both_legs": round(TAKER_FEE * 2 * 100, 4),
            "atr_used": round(atr_value, 2),
        }


# ══════════════════════════════════════════════════════════════
# ★ v7.0: TRAIN SINGLE TIMEFRAME
# ══════════════════════════════════════════════════════════════
def train_pattern_model_v7(df_raw, tf, n_splits=6):
    cfg             = TIMEFRAME_CONFIG[tf]
    min_candles     = cfg.get('min_candles',     200)
    horizon         = cfg.get('horizon',         5)
    min_return      = cfg.get('min_return',      0.003)
    atr_target_mult = cfg.get('atr_target_mult', 1.5)
    use_simple_split = cfg.get('use_simple_split', False)
    use_htf_filter   = cfg.get('use_htf_filter',   True)
    min_ps           = cfg.get('min_pattern_score', 0.10)
    time_filter      = cfg.get('time_filter_hours', None)

    logging.info("=" * 60)
    logging.info(f"فريم: {tf} | BTC Patterns v7.0 | Dynamic Ensemble | OOS={OOS_PCT:.0%}")
    logging.info("=" * 60)

    # Resample
    df_tf = (
        df_raw.set_index('datetime')
        .resample(cfg['resample_rule'])
        .agg({'open': 'first', 'high': 'max',
              'low':  'min',  'close': 'last', 'volume': 'sum'})
        .dropna()
        .reset_index()
    )
    if 'datetime' not in df_tf.columns:
        df_tf = df_tf.rename(columns={df_tf.columns[0]: 'datetime'})

    if cfg.get('years'):
        cut   = df_tf['datetime'].max() - timedelta(days=365 * cfg['years'])
        df_tf = df_tf[df_tf['datetime'] >= cut].copy()

    logging.info(
        f"شموع: {len(df_tf):,} | "
        f"{df_tf['datetime'].min().date()} → {df_tf['datetime'].max().date()}"
    )

    if len(df_tf) < min_candles:
        logging.error(f"❌ بيانات غير كافية ({len(df_tf)} < {min_candles})")
        return None

    # ★ v7.0: احجز الـ OOS من البداية قبل أي معالجة
    oos_cut    = int(len(df_tf) * (1 - OOS_PCT))
    df_train_p = df_tf.iloc[:oos_cut].copy().reset_index(drop=True)
    df_oos_raw = df_tf.iloc[oos_cut:].copy().reset_index(drop=True)

    logging.info(
        f"★ OOS محجوز: {len(df_oos_raw)} شمعة "
        f"({df_oos_raw['datetime'].min().date()} → {df_oos_raw['datetime'].max().date()})"
    )

    # Feature engineering على training pool فقط
    fe      = PatternFeatureEngineer()
    df_feat = fe.create_features(df_train_p, config=cfg, shift=True)

    # Target
    df_tgt = fe.create_target(
        df_feat, horizon=horizon,
        min_return=min_return,
        atr_target_mult=atr_target_mult
    )

    if df_tgt.empty or len(df_tgt) < max(min_candles // 2, 25):
        logging.error(f"❌ بيانات غير كافية بعد الـ target: {len(df_tgt)} صف")
        return None

    # ★ v7.0: تحقق من direction balance
    buy_r  = (df_tgt['target'] == 1).mean()
    sell_r = (df_tgt['target'] == 0).mean()
    logging.info(f"Direction balance — Buy: {buy_r:.1%} | Sell: {sell_r:.1%}")
    if abs(buy_r - sell_r) > 0.3:
        logging.warning(f"⚠️ [{tf}] Target غير متوازن — قد يظهر bias في النتائج")

    df_tgt = df_tgt.reset_index(drop=True)
    X      = df_tgt[fe.feature_columns]
    y      = df_tgt['target']
    logging.info(f"X.shape={X.shape} | Buy={buy_r:.1%} | Sell={sell_r:.1%}")

    # Pattern stats
    stats = {}
    for pc in ['hs_pattern', 'hs_valid', 'double_pattern', 'double_valid',
               'triple_pattern', 'triple_valid', 'triangle_pattern', 'wedge_pattern',
               'flag_pattern', 'cup_handle_pattern', 'cup_handle_valid',
               'rectangle_pattern', 'harmonic_pattern', 'harmonic_valid',
               'elliott_wave3', 'elliott_extended_w3', 'elliott_impulse',
               'elliott_corrective', 'elliott_valid',
               'pattern_cluster', 'strong_cluster', 'pattern_valid', 'kc_squeeze']:
        if pc in df_tgt.columns:
            stats[pc] = int((df_tgt[pc] > 0).sum())
    logging.info(f"Pattern stats: {stats}")

    # Walk-Forward أو Simple Split
    n_folds = 4 if tf == "1W" else n_splits

    if use_simple_split:
        logging.info(f"  [{tf}] استخدام Simple Time Split بدل WF (بيانات قليلة)")
        try:
            fold_res, best_ensemble, best_fe = simple_time_split_v7(
                X, y, df_tgt, cfg, train_ratio=0.70, tf_name=tf
            )
        except Exception as e:
            logging.error(f"❌ Simple Split فشل: {e}")
            return None
    else:
        try:
            fold_res, best_ensemble, best_fe = walk_forward_expanding_v7(
                X, y, df_tgt, cfg, n_splits=n_folds,
                purge_pct=None, tf_name=tf
            )
        except ValueError as e:
            logging.error(f"❌ Walk-Forward فشل: {e}")
            return None

    if not fold_res:
        logging.error(f"❌ لا توجد folds — {tf}")
        return None

    # Metrics
    wf_accs = [f['accuracy']      for f in fold_res]
    wf_aucs = [f.get('auc', 0.5)  for f in fold_res]
    wf_prs  = [f['precision']     for f in fold_res]
    wf_pfs  = [f['backtest']['profit_factor'] for f in fold_res
               if f.get('backtest') and f['backtest'].get('profit_factor')]
    wf_wrs  = [f['backtest']['win_rate']      for f in fold_res
               if f.get('backtest') and f['backtest'].get('win_rate')]
    wf_trd  = [f['backtest']['total_trades']  for f in fold_res
               if f.get('backtest') and f['backtest'].get('total_trades')]
    wf_shr  = [f['backtest'].get('sharpe_proxy', 0) for f in fold_res
               if f.get('backtest')]
    wf_dir  = [f['backtest'].get('direction_balance', 0.5) for f in fold_res
               if f.get('backtest')]

    if wf_accs: logging.info(f"  Avg Acc: {np.mean(wf_accs):.2%} ± {np.std(wf_accs):.2%}")
    if wf_aucs: logging.info(f"  Avg AUC: {np.mean(wf_aucs):.3f} ± {np.std(wf_aucs):.3f}")
    if wf_trd:  logging.info(f"  Total Trades: {sum(wf_trd)} | Avg/fold: {np.mean(wf_trd):.1f}")
    if wf_pfs:  logging.info(f"  Avg PF: {np.mean(wf_pfs):.2f}")
    if wf_shr:  logging.info(f"  Avg Sharpe: {np.mean(wf_shr):.3f}")
    if wf_dir:  logging.info(f"  Direction balance: {np.mean(wf_dir):.2f} (0=all sell, 1=all buy)")

    # Final Ensemble على كامل البيانات
    fef_full = PatternFeatureEngineer()
    fef_full.fit_scaler(X)
    Xfs = fef_full.transform(X)
    neg_f = (y == 0).sum()
    pos_f = (y == 1).sum()
    spw_f = neg_f / pos_f if pos_f else 1.0

    final_models, _, final_dyn_weights = train_ensemble_v7(Xfs, y, Xfs, y, spw_f)
    final_ensemble = EnsemblePredictor(final_models, weights=final_dyn_weights)
    fef_full.feature_columns = fe.feature_columns

    # Feature importance من XGBoost
    xgb_model = final_models.get('xgb')
    feat_imp   = pd.DataFrame({'feature': fef_full.feature_columns, 'importance': 0.0})
    if xgb_model is not None and hasattr(xgb_model, 'feature_importances_'):
        feat_imp['importance'] = xgb_model.feature_importances_
    feat_imp = feat_imp.sort_values('importance', ascending=False).head(20)
    logging.info("أهم 20 ميزة:\n" + feat_imp.to_string(index=False))

    # ★ v7.0: Holdout (آخر 15% من training pool)
    hs          = int(len(df_tgt) * 0.85)
    min_pq_hold = cfg.get('min_pattern_quality', 0.08)

    hmbt = Backtester(
        require_valid=False,
        min_pattern_quality=min_pq_hold,
        min_pattern_score=min_ps,
        use_htf_filter=use_htf_filter,
        time_filter_hours=time_filter,
        confidence_threshold=cfg.get('confidence_threshold', 0.52),
        atr_sl_mult=cfg.get('atr_sl_mult', ATR_SL_MULT),
        atr_tp_mult=cfg.get('atr_tp_mult', ATR_TP_MULT),
        is_walk_forward=False
    ).run(
        df_tgt.iloc[hs:].copy(), final_ensemble, fef_full,
        fef_full.feature_columns, cfg, use_fold=False, tf=tf
    )

    if hmbt:
        logging.info(
            f"✅ Holdout | T:{hmbt['total_trades']} WR:{hmbt['win_rate']}% "
            f"PF:{hmbt['profit_factor']} Sharpe:{hmbt.get('sharpe_proxy',0):.3f} "
            f"Buy/Sell:{hmbt.get('direction_balance',0.5):.2f}"
        )

    # ★ v7.0: OOS Test المستقل الحقيقي
    oos_bt = {}
    try:
        df_oos_feat = fe.create_features(df_oos_raw, config=cfg, shift=False)
        if not df_oos_feat.empty:
            oos_bt = Backtester(
                require_valid=False,
                min_pattern_quality=min_pq_hold,
                min_pattern_score=min_ps,
                use_htf_filter=use_htf_filter,
                time_filter_hours=time_filter,
                confidence_threshold=cfg.get('confidence_threshold', 0.52),
                atr_sl_mult=cfg.get('atr_sl_mult', ATR_SL_MULT),
                atr_tp_mult=cfg.get('atr_tp_mult', ATR_TP_MULT),
                is_walk_forward=False
            ).run(
                df_oos_feat, final_ensemble, fef_full,
                fef_full.feature_columns, cfg, use_fold=False, tf=tf
            )
            if oos_bt:
                pf_oos = oos_bt.get('profit_factor', 0)
                status = "✅" if pf_oos >= 1.0 else "❌"
                logging.info(
                    f"{status} OOS حقيقي | T:{oos_bt['total_trades']} "
                    f"WR:{oos_bt['win_rate']}% PF:{pf_oos} "
                    f"Sharpe:{oos_bt.get('sharpe_proxy',0):.3f}"
                )
                if pf_oos < 1.0:
                    logging.warning(
                        f"⚠️ [{tf}] OOS PF < 1.0 — الموديل قد لا يعمل على بيانات مستقبلية!"
                    )
    except Exception as e:
        logging.warning(f"OOS test error [{tf}]: {e}")

    # ATR Demo
    sample_price = float(df_tgt['close'].iloc[-1])
    sample_atr   = float(df_tgt['atr'].iloc[-1]) if 'atr' in df_tgt.columns else sample_price * 0.015
    sample_tgt   = calculate_btc_targets(sample_price, direction=1, atr_value=sample_atr)
    logging.info(
        f"📊 أهداف ATR عند {sample_price:,.0f}$ | ATR={sample_atr:,.0f}$\n"
        f"   SL:{sample_tgt['sl']:,.2f} TP1:{sample_tgt['tp1']:,.2f} TP:{sample_tgt['tp_full']:,.2f} R:R={sample_tgt['rr']}"
    )

    # حفظ الموديل
    os.makedirs(MODELS_DIR, exist_ok=True)
    mpath = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf}_patterns_v7.pkl")
    joblib.dump({
        'ensemble':            final_ensemble,
        'models':              final_models,
        'scaler':              fef_full.scaler,
        'feature_columns':     fef_full.feature_columns,
        'config':              cfg,
        'feature_importance':  feat_imp.to_dict('records'),
        'version':             'v7.0',
        'atr_sl_mult':         ATR_SL_MULT,
        'atr_tp_mult':         ATR_TP_MULT,
        'dynamic_weights':     final_dyn_weights,
        'direction_balance':   {'buy': float(buy_r), 'sell': float(sell_r)},
        'oos_result':          oos_bt,
        'btc_targets_info': {
            'note':      'SL/TP based on ATR',
            'tp1_ratio': BTC_TP1_RATIO, 'tp2_ratio': BTC_TP2_RATIO,
            'tp3_ratio': BTC_TP3_RATIO, 'tp4_ratio': BTC_TP4_RATIO,
            'sl_ratio':  BTC_SL_RATIO,  'taker_fee': TAKER_FEE,
        }
    }, mpath)

    # حفظ الـ metrics
    jpath = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf}_patterns_v7_metrics.json")
    with open(jpath, 'w', encoding='utf-8') as f:
        json.dump({
            'version':   'v7.0_btc_dynamic_ensemble',
            'symbol':    'BTC/USD',
            'timeframe': tf,
            'atr_config': {
                'atr_sl_mult': ATR_SL_MULT,
                'atr_tp_mult': ATR_TP_MULT,
                'rr':          round(ATR_TP_MULT / ATR_SL_MULT, 2),
                'spread_bps':  SPREAD_BPS,
                'taker_fee':   TAKER_FEE,
                'note':        'v7.0: Dynamic ensemble + OOS + Circuit Breaker + Balanced target',
            },
            'target_balance': {'buy_pct': round(float(buy_r), 4), 'sell_pct': round(float(sell_r), 4)},
            'oos_reserved': {
                'pct':   OOS_PCT,
                'start': str(df_oos_raw['datetime'].min()) if len(df_oos_raw) > 0 else '',
                'end':   str(df_oos_raw['datetime'].max()) if len(df_oos_raw) > 0 else '',
            },
            'walk_forward': {
                'n_folds':           len(fold_res),
                'method':            'simple_split' if use_simple_split else 'expanding_wf',
                'avg_accuracy':      round(float(np.mean(wf_accs)),  4) if wf_accs else 0,
                'std_accuracy':      round(float(np.std(wf_accs)),   4) if wf_accs else 0,
                'avg_auc':           round(float(np.mean(wf_aucs)),  4) if wf_aucs else 0,
                'avg_precision':     round(float(np.mean(wf_prs)),   4) if wf_prs  else 0,
                'avg_win_rate':      round(float(np.mean(wf_wrs)),   2) if wf_wrs  else 0,
                'avg_profit_factor': round(float(np.mean(wf_pfs)),   2) if wf_pfs  else 0,
                'avg_sharpe_proxy':  round(float(np.mean(wf_shr)),   3) if wf_shr  else 0,
                'avg_dir_balance':   round(float(np.mean(wf_dir)),   3) if wf_dir  else 0,
                'total_trades_wf':   sum(wf_trd) if wf_trd else 0,
                'avg_trades_fold':   round(float(np.mean(wf_trd)),   1) if wf_trd  else 0,
                'consistency_pct':   round(
                    sum(1 for p in wf_pfs if p > 1) / max(len(wf_pfs), 1) * 100, 1
                ),
                'folds': fold_res
            },
            'holdout_last_15pct': hmbt,
            'oos_independent':    oos_bt,
            'config':             cfg,
            'pattern_stats':      stats,
            'training_samples':   len(df_tgt),
            'features':           len(fef_full.feature_columns),
            'feature_importance': feat_imp.to_dict('records'),
            'ensemble_info': {
                'models':  list(final_models.keys()),
                'weights': final_dyn_weights,
                'note':    'Dynamic weights based on validation AUC per fold'
            }
        }, f, indent=2, default=str, ensure_ascii=False)

    logging.info(f"💾 محفوظ: {mpath}")
    return final_ensemble, fef_full


# ══════════════════════════════════════════════════════════════
# TRAIN ALL TIMEFRAMES
# ══════════════════════════════════════════════════════════════
def train_all_pattern_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("train_patterns_btc.py  v7.0  — BTC/USD")
    logging.info("تحسينات v7.0:")
    logging.info("  ① Target متوازن Buy/Sell (يحل SELL bias)")
    logging.info("  ② Dynamic Ensemble Weights (بناءً على AUC الفعلي)")
    logging.info("  ③ LightGBM كرابع موديل في الـ ensemble")
    logging.info("  ④ Circuit Breaker: 5 losses → 10 bars cooling")
    logging.info("  ⑤ OOS Test مستقل (آخر 10% محجوزة من البداية)")
    logging.info("  ⑥ Multi-confirmation filter في الـ Backtester")
    logging.info("  ⑦ Regime-aware thresholds (ADX-based)")
    logging.info("  ⑧ 1d: atr_target_mult=1.0, years=12, min_pq=0.05")
    logging.info("  ⑨ 1W: Simple Split بدل WF + zigzag=0.012")
    logging.info("  ⑩ Trailing Stop بعد TP1 (بدل breakeven فوري)")
    logging.info(f"ATR: SL×{ATR_SL_MULT} | TP×{ATR_TP_MULT} | R:R={ATR_TP_MULT/ATR_SL_MULT:.2f}")
    logging.info(f"Output: {MODELS_DIR}/")
    logging.info("=" * 70)

    df_raw = get_or_update_data(force_refresh=force_refresh_data)
    if df_raw.empty:
        logging.error("❌ فشل تحميل الداتا")
        return

    logging.info(
        f"الداتا: {len(df_raw):,} شمعة | "
        f"{df_raw['datetime'].min()} → {df_raw['datetime'].max()}"
    )

    results    = {}
    wf_summary = {}

    for tf in TIMEFRAME_CONFIG:
        logging.info(f"\n{'─'*50}\nفريم: {tf}")
        try:
            res = train_pattern_model_v7(df_raw, tf, n_splits=6)
            results[tf] = "✅ نجح" if res else "❌ فشل"
        except Exception as e:
            logging.error(f"❌ خطأ في الفريم {tf}: {e}", exc_info=True)
            results[tf] = f"❌ خطأ: {e}"

        jpath = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf}_patterns_v7_metrics.json")
        if os.path.exists(jpath):
            with open(jpath, encoding='utf-8') as f:
                m  = json.load(f)
                wf = m.get('walk_forward', {})
                oos = m.get('oos_independent', {})
                wf_summary[tf] = {
                    'folds':        wf.get('n_folds',            '-'),
                    'avg_acc':      wf.get('avg_accuracy',       '-'),
                    'avg_auc':      wf.get('avg_auc',            '-'),
                    'avg_pf':       wf.get('avg_profit_factor',  '-'),
                    'total_trades': wf.get('total_trades_wf',    '-'),
                    'avg_trades':   wf.get('avg_trades_fold',    '-'),
                    'consistency':  wf.get('consistency_pct',    '-'),
                    'sharpe':       wf.get('avg_sharpe_proxy',   '-'),
                    'dir_balance':  wf.get('avg_dir_balance',    '-'),
                    'oos_pf':       oos.get('profit_factor',     '-'),
                    'oos_trades':   oos.get('total_trades',      '-'),
                }

    logging.info("\n" + "=" * 70)
    logging.info("الملخص النهائي — BTC Pattern Models v7.0:")
    logging.info(f"  {'TF':<8} {'Status':<18} {'AUC':<7} {'PF':<6} {'OOS_PF':<8} {'Trades':<9} {'DirBal'}")
    for tf, st in results.items():
        wf = wf_summary.get(tf, {})
        oos_pf = wf.get('oos_pf', '-')
        oos_ok = "✅" if isinstance(oos_pf, (int, float)) and oos_pf >= 1.0 else "❌"
        logging.info(
            f"  {tf:<8} {st:<18} "
            f"{str(wf.get('avg_auc','-')):<7} "
            f"{str(wf.get('avg_pf','-')):<6} "
            f"{oos_ok}{str(oos_pf):<7} "
            f"{str(wf.get('total_trades','-')):<9} "
            f"{str(wf.get('dir_balance','-'))}"
        )

    logging.info(f"\n📊 v7.0: Dynamic Ensemble + OOS + Circuit Breaker")
    logging.info(f"   ATR_SL×{ATR_SL_MULT} | ATR_TP×{ATR_TP_MULT} | R:R={ATR_TP_MULT/ATR_SL_MULT:.2f}")
    logging.info(f"   OOS محجوز: {OOS_PCT:.0%} من كل فريم")
    logging.info(f"\nجميع الملفات في: {MODELS_DIR}/")
    logging.info("=" * 70)


# ══════════════════════════════════════════════════════════════
# LIVE SIGNAL GENERATOR
# ══════════════════════════════════════════════════════════════
def load_model_v7(tf: str):
    mpath = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf}_patterns_v7.pkl")
    if not os.path.exists(mpath):
        raise FileNotFoundError(f"Model not found: {mpath}")
    data = joblib.load(mpath)
    return data['ensemble'], data['scaler'], data['feature_columns'], data['config']


def generate_live_signal(df_recent: pd.DataFrame, tf: str) -> dict:
    """
    v7.0: Live signal مع direction balance check
    """
    try:
        ensemble, scaler, feat_cols, cfg = load_model_v7(tf)

        fe = PatternFeatureEngineer()
        fe.scaler          = scaler
        fe.feature_columns = feat_cols
        fe.scaler_fitted   = True

        df_feat = fe.create_features(df_recent, config=cfg, shift=False)
        if df_feat.empty:
            return {"signal": "NO SIGNAL", "reason": "بيانات غير كافية"}

        last_row = df_feat.iloc[[-1]]
        missing  = [c for c in feat_cols if c not in last_row.columns]
        for c in missing:
            last_row[c] = 0.0

        X_live = last_row[feat_cols].fillna(0).replace([np.inf, -np.inf], 0)
        X_s    = scaler.transform(X_live)
        prob   = ensemble.predict_proba(X_s)[0, 1]

        bth = cfg.get('buy_threshold',  0.52)
        sth = cfg.get('sell_threshold', 0.48)

        # ★ v7.0: Regime-aware thresholds في الـ live
        adx_live = float(last_row.get('adx', pd.Series([20])).iloc[0])
        if adx_live > 25:
            bth = max(0.48, bth - 0.02)
            sth = min(0.52, sth + 0.02)
        elif adx_live < 15:
            bth = min(0.60, bth + 0.03)
            sth = max(0.40, sth - 0.03)

        direction = 1 if prob > bth else (-1 if prob < sth else 0)
        signal    = "BUY" if direction == 1 else ("SELL" if direction == -1 else "HOLD")

        entry_price = float(df_feat['close'].iloc[-1])
        atr_value   = float(df_feat['atr'].iloc[-1]) if 'atr' in df_feat.columns else entry_price * 0.015
        targets     = calculate_btc_targets(entry_price, direction, atr_value) if direction != 0 else {}

        # ★ v7.0: multi-confirm check للـ live
        multi_c = float(last_row.get('multi_confirm_score', pd.Series([0])).iloc[0])
        min_ps  = cfg.get('min_pattern_score', 0.10)
        p_score = float(last_row.get('pattern_score', pd.Series([0])).iloc[0])

        confidence_label = (
            "STRONG 🔥" if prob > 0.75 else
            "MEDIUM 📊" if prob > 0.60 else
            "WEAK ⚠️"  if prob > 0.52 else "LOW ❌"
        )

        warnings = []
        if p_score < min_ps:
            warnings.append(f"pattern_score منخفض ({p_score:.3f} < {min_ps})")
        if multi_c < 1:
            warnings.append(f"multi_confirm منخفض ({multi_c:.0f})")

        return {
            "signal":            signal,
            "direction":         direction,
            "probability":       round(float(prob), 4),
            "confidence_label":  confidence_label,
            "pattern_quality":   round(float(last_row.get('pattern_quality', pd.Series([0])).iloc[0]), 4),
            "pattern_score":     round(p_score, 4),
            "pattern_valid":     int(last_row.get('pattern_valid', pd.Series([0])).iloc[0]),
            "multi_confirm":     int(multi_c),
            "htf_trend":         int(last_row.get('htf_trend', pd.Series([0])).iloc[0]),
            "adx":               round(adx_live, 2),
            "bull_market":       bool(last_row.get('bull_market', pd.Series([0])).iloc[0] > 0.5),
            "effective_bth":     round(bth, 3),
            "effective_sth":     round(sth, 3),
            "targets":           targets,
            "warnings":          warnings,
            "timeframe":         tf,
            "timestamp":         str(df_feat['datetime'].iloc[-1]),
            "version":           "v7.0_dynamic_ensemble"
        }
    except Exception as e:
        logging.error(f"Live signal error [{tf}]: {e}")
        return {"signal": "ERROR", "reason": str(e)}


# ══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    logging.info("=" * 70)
    logging.info("⚠️  v7.0 — Dynamic Ensemble + OOS + Circuit Breaker")
    logging.info("   التحسينات الرئيسية:")
    logging.info("   ① Target متوازن (Buy/Sell) — يحل SELL bias")
    logging.info("   ② Dynamic weights بناءً على AUC كل موديل")
    logging.info("   ③ OOS مستقل (10%) — المقياس الحقيقي")
    logging.info("   ④ Circuit Breaker — يوقف التداول بعد 5 losses")
    logging.info("   ⑤ LightGBM كرابع موديل (لو مثبّت)")
    logging.info("   شغّل fetch_data_btc.py أولاً لو أول مرة")
    logging.info("=" * 70)
    train_all_pattern_models(force_refresh_data=False)

In [4]:
# ============================================================
# download_models


import os
import shutil
from google.colab import files

MODELS_DIR = "models/btc_patterns"
ZIP_NAME   = "BTC_CLassic_models.zip"

# ── تحقق من وجود الفولدر ──
if not os.path.exists(MODELS_DIR):
    print(f"❌ الفولدر '{MODELS_DIR}' مش موجود!")
    print("تأكد إن silver_patterns_v35.py اشتغل وحفظ الملفات أولاً.")
else:
    # ── اعرض الملفات الموجودة ──
    all_files = os.listdir(MODELS_DIR)
    pkl_files  = [f for f in all_files if f.endswith('.pkl')]
    json_files = [f for f in all_files if f.endswith('.json')]

    print(f"📁 محتوى '{MODELS_DIR}':")
    print(f"   .pkl  (موديلات): {len(pkl_files)}")
    print(f"   .json (مقاييس):  {len(json_files)}")
    print()
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<45} {size/1024:.1f} KB")

    # ── اضغط الفولدر ──
    print(f"\n📦 جاري ضغط الفولدر ...")
    shutil.make_archive(
        ZIP_NAME.replace('.zip', ''),  # اسم الملف
        'zip',                          # نوع الضغط
        '.',                            # المجلد الأساسي
        MODELS_DIR                      # الفولدر المراد ضغطه
    )

    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"✅ تم إنشاء: {ZIP_NAME} ({size_mb:.1f} MB)")

    # ── نزّل الملف ──
    print(f"⬇️  جاري التحميل ...")
    files.download(ZIP_NAME)
    print(f"✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.")

📁 محتوى 'models/btc_patterns':
   .pkl  (موديلات): 8
   .json (مقاييس):  8

   BTCUSD_15min_patterns_v7.pkl                  4629.0 KB
   BTCUSD_15min_patterns_v7_metrics.json         29.0 KB
   BTCUSD_1W_patterns_v7.pkl                     601.0 KB
   BTCUSD_1W_patterns_v7_metrics.json            7.1 KB
   BTCUSD_1d_patterns_v7.pkl                     2317.9 KB
   BTCUSD_1d_patterns_v7_metrics.json            19.3 KB
   BTCUSD_1h_patterns_v7.pkl                     4091.1 KB
   BTCUSD_1h_patterns_v7_metrics.json            34.5 KB
   BTCUSD_2h_patterns_v7.pkl                     3679.4 KB
   BTCUSD_2h_patterns_v7_metrics.json            32.0 KB
   BTCUSD_45min_patterns_v7.pkl                  4320.0 KB
   BTCUSD_45min_patterns_v7_metrics.json         32.0 KB
   BTCUSD_4h_patterns_v7.pkl                     2950.8 KB
   BTCUSD_4h_patterns_v7_metrics.json            31.9 KB
   BTCUSD_5min_patterns_v7.pkl                   4845.6 KB
   BTCUSD_5min_patterns_v7_metrics.json          25.9 K

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.
